# Data Engineering Interview Questions & Detailed Answers

A comprehensive collection of generic and scenario-based data engineering interview questions commonly asked at companies like Amazon, Google, Meta, Netflix, Uber, and Airbnb. Each question includes a detailed answer covering architecture, trade-offs, and best practices.

---

## 📖 Generic Data Engineering Interview Questions

Fundamental, broad-scope data engineering questions that test core knowledge and understanding. These are commonly asked at companies like Amazon, Google, Microsoft, Netflix, LinkedIn, and Uber to assess foundational data engineering concepts.

### G1: What is the difference between OLTP and OLAP systems?

**Answer:**

| Criterion | OLTP (Online Transaction Processing) | OLAP (Online Analytical Processing) |
|---|---|---|
| **Purpose** | Day-to-day business operations | Business intelligence and analysis |
| **Query type** | Simple, short (INSERT, UPDATE, DELETE, point SELECT) | Complex, long (aggregations, joins, scans) |
| **Data volume** | Current/operational data | Historical, aggregated data |
| **Users** | Application users, customers, operations staff | Analysts, managers, executives |
| **Database design** | Normalized (3NF) for write efficiency | Denormalized (star/snowflake) for read efficiency |
| **Transaction volume** | High (thousands per second) | Low (few complex queries) |
| **Response time** | Milliseconds | Seconds to minutes |
| **Examples** | Banking app, e-commerce checkout, ERP | Sales reporting, customer segmentation, forecasting |

**Key differences:**
- OLTP systems prioritize **consistency, availability, and low latency** for individual transactions. They handle concurrent reads/writes with strict ACID compliance.
- OLAP systems prioritize **query performance on large datasets**. They are optimized for reading and aggregating massive amounts of data, not for transactional writes.
- OLTP databases are typically row-store (PostgreSQL, MySQL, Oracle). OLAP databases are typically column-store (Snowflake, BigQuery, Redshift, Databricks).
- In modern architectures, OLTP data is replicated or streamed (via CDC) into an OLAP system (data warehouse/lakehouse) for analytics, keeping operational and analytical workloads separate.

### G2: Explain normalization and denormalization. When would you use each?

**Answer:**

**Normalization** is the process of organizing data to reduce redundancy and improve data integrity. Data is split into multiple related tables using foreign keys.

**Normal Forms (most common):**
- **1NF:** Each column contains atomic (indivisible) values. No repeating groups.
- **2NF:** 1NF + all non-key attributes are fully dependent on the entire primary key (no partial dependency). Relevant for composite keys.
- **3NF:** 2NF + no transitive dependency (non-key attributes depend only on the primary key, not on other non-key attributes).
- **BCNF:** A stronger version of 3NF where every determinant is a candidate key.

**Denormalization** is the process of combining tables to reduce joins and improve read performance, at the cost of data redundancy.

**When to use normalization:**
- OLTP systems with frequent INSERT/UPDATE/DELETE operations.
- When data integrity is critical (avoid anomalies from redundant data).
- When storage is a concern (less redundancy = less storage).
- When the schema changes frequently (normalized data is easier to maintain).

**When to use denormalization:**
- OLAP/data warehousing with heavy read workloads.
- When query performance is more important than storage.
- When joins are expensive and slow down dashboards/reports.
- When the data is relatively static (few updates).

**Practical example:**
- Normalized (OLTP): `orders`, `customers`, `products` as separate tables, joined on foreign keys.
- Denormalized (OLAP): A single `sales_summary` table with customer name, product name, and order details pre-joined.

**Trade-off:** Normalization optimizes for writes (fewer duplicates to update). Denormalization optimizes for reads (fewer joins to execute). Most data warehouses use a hybrid: normalized in the raw/silver layer, denormalized in the gold/serving layer.

### G4: What is the difference between structured, semi-structured, and unstructured data?

**Answer:**

| Type | Description | Examples | Storage |
|---|---|---|---|
| **Structured** | Data with a predefined schema (rows and columns) | Relational tables, CSV with headers | RDBMS, data warehouses, Parquet |
| **Semi-structured** | Data with a flexible, self-describing schema (tags, markers) | JSON, XML, Avro, YAML, logs | Data lakes, NoSQL (MongoDB, Cassandra) |
| **Unstructured** | Data with no predefined format or schema | Images, videos, audio, PDFs, free text | Object storage (S3), file systems |

**Key distinctions:**
- **Structured data** is easy to query with SQL. Schema is enforced at write time. Fixed schema means all records conform to the same structure.
- **Semi-structured data** has a schema embedded within the data itself. Each record may have different fields. Can be queried with SQL if parsed into a structured format (e.g., Delta Lake reads JSON as VARIANT columns). Flexible but less efficient for storage and querying.
- **Unstructured data** requires processing (NLP, OCR, ML) to extract meaning. Cannot be queried directly with SQL. Stored as binary blobs.

**In modern data engineering:**
- All three types coexist in a lakehouse: structured in Delta tables, semi-structured as VARIANT columns or raw files, unstructured in object storage.
- Spark/Databricks can process all three: SQL on structured DataFrames, `from_json` for semi-structured, ML libraries for unstructured.
- The trend is toward storing semi-structured data in structured formats (Parquet/Delta) with schema evolution support.

### G5: What is schema-on-read vs schema-on-write?

**Answer:**

**Schema-on-write** (traditional RDBMS, data warehouse):
- The schema is defined before data is loaded. Data must conform to the schema to be accepted.
- If data doesn't match the schema, the load fails.
- Benefit: Data is guaranteed to be clean and consistent. Queries are fast because the engine knows the structure.
- Drawback: Inflexible. Adding a new column requires a schema migration. Cannot store data that doesn't fit the existing schema.
- Examples: PostgreSQL, MySQL, Snowflake, BigQuery (traditional tables).

**Schema-on-read** (data lake):
- No schema is enforced at storage time. Data is stored as-is (raw files).
- The schema is applied when the data is read/queried. Different readers can interpret the same data differently.
- Benefit: Extremely flexible. Can store any data without upfront schema design. Easy to ingest new data sources.
- Drawback: No data quality guarantee. Queries may fail if data doesn't match expectations. No enforcement of constraints.
- Examples: Hive, Spark reading raw JSON/CSV files, Presto/Trino on S3.

**Modern approach (Lakehouse — best of both):**
- Delta Lake/Iceberg/Hudi provide **schema enforcement** (schema-on-write) on top of data lake storage.
- They also support **schema evolution** — you can add new columns without rewriting data.
- This gives you the flexibility of schema-on-read with the reliability of schema-on-write.

**When to use each:**
- Schema-on-write: When data quality and consistency are critical (financial, healthcare, regulatory).
- Schema-on-read: When exploring new data sources, rapid prototyping, or when schema is unpredictable.
- Lakehouse (both): Production data pipelines where you need flexibility + reliability.

### G6: What are the key responsibilities of a data engineer?

**Answer:**

A data engineer is responsible for the **infrastructure, pipelines, and platforms** that enable data-driven decision-making. Core responsibilities include:

**1. Data Ingestion and Pipeline Development**
- Build and maintain ETL/ELT pipelines that move data from source systems to storage/warehouse.
- Handle batch and real-time/streaming data ingestion.
- Ensure data is ingested reliably, with error handling and retry logic.

**2. Data Storage and Architecture**
- Design and implement data storage solutions (data lakes, warehouses, lakehouses).
- Choose appropriate file formats (Parquet, Delta, Avro) and partitioning strategies.
- Implement data modeling (star schema, snowflake, Data Vault, medallion architecture).

**3. Data Transformation and Processing**
- Clean, filter, deduplicate, and transform raw data into analysis-ready formats.
- Build aggregated tables and materialized views for fast querying.
- Implement business logic and data quality rules.

**4. Data Quality and Reliability**
- Implement data quality checks (completeness, accuracy, consistency, timeliness).
- Monitor pipeline health and set up alerting for failures.
- Ensure idempotent pipelines that can safely retry.

**5. Data Governance and Security**
- Implement access control (RBAC, column masking, row-level security).
- Manage PII detection and compliance (GDPR, CCPA, HIPAA).
- Maintain data lineage, cataloging, and documentation.

**6. Performance Optimization**
- Optimize query performance (partitioning, clustering, caching, indexing).
- Monitor and tune Spark/warehouse compute resources.
- Manage cost optimization (auto-scaling, tiered storage).

**7. Collaboration**
- Work with data scientists to provide clean, reliable data for ML models.
- Work with analysts to understand reporting requirements.
- Work with software engineers to integrate data pipelines with applications.

**Data Engineer vs Data Scientist vs Data Analyst:**
| Role | Focus | Skills |
|---|---|---|
| **Data Engineer** | Building pipelines, infrastructure, data platforms | Spark, Airflow, SQL, Python, cloud, distributed systems |
| **Data Scientist** | Building ML models, statistical analysis, predictions | Python, ML libraries, statistics, experiment design |
| **Data Analyst** | Answering business questions, dashboards, reporting | SQL, BI tools (Tableau, Looker), Excel, basic Python |

### G7: What is the difference between a data warehouse, data lake, and data mart?

**Answer:**

**Data Warehouse:**
- Centralized repository for structured, processed data optimized for SQL queries and BI.
- Stores historical data from multiple sources, transformed via ETL.
- Schema-on-write, ACID transactions, columnar storage.
- Examples: Snowflake, BigQuery, Redshift, Teradata.

**Data Lake:**
- Centralized repository for raw data in any format (structured, semi-structured, unstructured).
- Stores data as-is without processing. Schema-on-read.
- Cheap object storage (S3, ADLS). Handles any data type.
- Risk: Can become a "data swamp" without governance.

**Data Mart:**
- A subset of a data warehouse, focused on a specific business domain or department.
- Contains pre-aggregated, domain-specific data (e.g., sales data mart, finance data mart).
- Faster queries for department-specific use cases.
- Can be built from the data warehouse (dependent data mart) or directly from sources (independent data mart).

**Comparison:**

| Criterion | Data Warehouse | Data Lake | Data Mart |
|---|---|---|---|
| **Scope** | Enterprise-wide | Enterprise-wide | Department/domain |
| **Data type** | Structured only | All types | Structured (aggregated) |
| **Schema** | On-write | On-read | On-write |
| **Processing** | ETL (transformed) | Raw (no processing) | Pre-aggregated |
| **Users** | Analysts, BI tools | Data scientists, engineers | Department-specific users |
| **Cost** | High | Low | Medium |
| **Query speed** | Fast | Variable | Fastest (small, pre-aggregated) |

**Modern architecture:** Data lake + data warehouse = **Lakehouse** (Delta Lake, Iceberg). Data marts are built as Gold-layer tables in the lakehouse.

### G8: What is the difference between batch processing and real-time processing?

**Answer:**

**Batch Processing:**
- Data is collected over a period and processed in large batches (e.g., nightly, hourly).
- High throughput, low latency requirement.
- Example: Nightly ETL job that processes all orders from the day.
- Tools: Spark (batch mode), Airflow, Hadoop MapReduce, dbt.

**Real-Time (Stream) Processing:**
- Data is processed as it arrives, with minimal delay (milliseconds to seconds).
- Low latency, continuous processing.
- Example: Fraud detection that flags suspicious transactions within 5 seconds.
- Tools: Spark Structured Streaming, Flink, Kafka Streams, AWS Kinesis.

**Comparison:**

| Criterion | Batch | Real-Time |
|---|---|---|
| **Latency** | Minutes to hours | Milliseconds to seconds |
| **Throughput** | Very high | Moderate (per message) |
| **Data volume** | Large batches | Individual events |
| **Complexity** | Lower | Higher (state management, fault tolerance) |
| **Cost** | Lower (scheduled compute) | Higher (always-on compute) |
| **Use cases** | Reporting, ETL, ML training | Alerting, dashboards, fraud detection |
| **Ordering** | Not critical | Often critical (per key) |

**Lambda vs Kappa Architecture:**
- **Lambda:** Both batch and real-time paths. Batch for accuracy, real-time for latency. Two codebases to maintain.
- **Kappa:** Only real-time path. Replays historical data through the streaming engine. One codebase, but harder for complex historical processing.

**Modern trend:** The boundary is blurring. Delta Lake with Structured Streaming allows batch and streaming on the same table. Micro-batch streaming (Spark Structured Streaming) processes data in small batches (every few seconds), combining the simplicity of batch with near-real-time latency.

### G9: What is data replication? Explain synchronous vs asynchronous replication.

**Answer:**

**Data Replication** is the process of copying data from a primary database to one or more secondary databases for availability, disaster recovery, read scaling, or geographic distribution.

**Synchronous Replication:**
- The primary waits for the secondary to acknowledge the write before committing the transaction.
- The write is considered successful only after both primary and secondary have it.
- **Guarantee:** Zero data loss — secondary always has the latest data.
- **Trade-off:** Higher latency (adds round-trip time). If the secondary is slow or down, writes are blocked.
- **Use case:** Financial systems, critical data where data loss is unacceptable.

**Asynchronous Replication:**
- The primary commits the transaction immediately and sends the update to the secondary later.
- The secondary may lag behind the primary by some amount of time.
- **Guarantee:** Higher throughput, no write blocking.
- **Trade-off:** Potential data loss if the primary fails before the secondary catches up.
- **Use case:** Read scaling, geographic distribution, systems that can tolerate some data lag.

**Comparison:**

| Criterion | Synchronous | Asynchronous |
|---|---|---|
| **Data loss risk** | None | Possible (if primary fails before sync) |
| **Write latency** | Higher (waits for ack) | Lower (immediate) |
| **Throughput** | Lower | Higher |
| **Availability** | Lower (writes blocked if replica down) | Higher (writes proceed regardless) |
| **Use case** | Financial, critical data | Read scaling, DR, geo-distribution |

**Common replication patterns in data engineering:**
- **Active-Passive:** Primary handles writes, secondaries serve reads. Failover to secondary if primary fails.
- **Active-Active:** Multiple primaries accept writes. Requires conflict resolution (last-write-wins, CRDTs, or application-level).
- **CDC-based replication:** Use log-based CDC (Debezium) to replicate changes from OLTP to OLAP systems asynchronously.
- **Multi-region replication:** Replicate data across cloud regions for disaster recovery and low-latency global access.

**In cloud data warehouses:**
- Snowflake: Automatic replication across regions/accounts (asynchronous).
- BigQuery: Cross-region dataset replication.
- Databricks: Delta Lake replication via Delta Live Tables or CDC to replicate data across regions.

### G10: What is the difference between a primary key, foreign key, and surrogate key?

**Answer:**

**Primary Key (PK):**
- A column (or set of columns) that uniquely identifies each row in a table.
- Must be unique and non-null.
- Example: `customer_id` in a `customers` table.
- Purpose: Entity integrity — ensures no duplicate rows.

**Foreign Key (FK):**
- A column (or set of columns) in one table that references the primary key of another table.
- Establishes a relationship between two tables.
- Can be null (optional relationship) or non-null (mandatory relationship).
- Example: `customer_id` in an `orders` table references `customer_id` in the `customers` table.
- Purpose: Referential integrity — ensures related data exists.

**Surrogate Key (SK):**
- An artificial, system-generated key (usually an integer or UUID) that has no business meaning.
- Used as the primary key instead of a natural/business key.
- Example: `customer_sk = 1, 2, 3...` instead of using `customer_email` as the key.
- Purpose:
  - Decouples the key from business logic (business keys can change, surrogate keys don't).
  - Essential for SCD Type 2 (each version of a dimension row gets a unique surrogate key).
  - Integer surrogate keys are faster for joins than string-based natural keys.
  - Avoids issues with composite natural keys.

**Comparison:**

| Key Type | Source | Uniqueness | Business Meaning | Use Case |
|---|---|---|---|---|
| **Primary Key** | Business or generated | Unique, non-null | May have meaning | Identify rows in a table |
| **Foreign Key** | References PK of another table | May have duplicates | None | Establish relationships |
| **Surrogate Key** | System-generated | Unique, non-null | None | Replace natural keys, SCD Type 2 |

**In data warehousing:**
- Fact tables use surrogate keys to join with dimension tables.
- This allows the dimension to change (SCD Type 2) without breaking the fact table.
- The fact table's `customer_key` points to a specific version of the customer, enabling point-in-time analysis.

### G11: What is data governance and why is it important?

**Answer:**

**Data Governance** is the framework of policies, processes, and technologies that ensure data is **available, usable, secure, and compliant** throughout its lifecycle.

**Core pillars:**

1. **Data Cataloging and Discovery:**
   - A searchable inventory of all data assets (tables, columns, pipelines).
   - Includes metadata: descriptions, owners, quality scores, lineage.
   - Tools: Unity Catalog, DataHub, Amundsen, Collibra.

2. **Access Control and Security:**
   - Role-based access control (RBAC): who can read/write what data.
   - Column-level masking: hide PII from unauthorized users.
   - Row-level security: filter rows based on user attributes.
   - Audit logging: track who accessed what data and when.

3. **Data Quality Management:**
   - Define and enforce data quality rules (completeness, accuracy, consistency).
   - Monitor quality metrics and alert on violations.
   - Data contracts between producers and consumers.

4. **Compliance and Privacy:**
   - GDPR, CCPA, HIPAA, SOX compliance.
   - PII detection and classification.
   - Right to be forgotten (GDPR): ability to delete user data across all systems.
   - Data retention policies: how long to keep data, when to purge.

5. **Data Lineage:**
   - Track data from origin to destination through all transformations.
   - Impact analysis: what breaks if a source table changes?
   - Root cause analysis: where did bad data come from?

6. **Ownership and Stewardship:**
   - Every data asset has a documented owner.
   - Data stewards ensure quality and compliance for their domain.
   - Clear escalation paths for data issues.

**Why it matters:**
- **Trust:** Stakeholders trust data when they know its origin, quality, and governance.
- **Compliance:** Regulatory fines for non-compliance can be millions of dollars.
- **Efficiency:** Engineers spend less time searching for data and resolving issues.
- **Risk mitigation:** Prevents data leaks, unauthorized access, and costly errors.
- **Democratization:** Enables self-service analytics while maintaining guardrails.

**In Databricks (Unity Catalog):**
- Unified governance across all data assets.
- Catalog -> Schema -> Table hierarchy with inherited permissions.
- Column-level lineage, tagging, and masking.
- Audit logs in `system.access.audit`.

### G12: What are the common data quality dimensions?

**Answer:**

Data quality is assessed across six key dimensions:

1. **Completeness:** Are all required fields populated?
   - Check: `SELECT COUNT(*) FROM table WHERE required_field IS NULL` -- should be 0.
   - Example: Every order must have a `customer_id`.

2. **Accuracy:** Does the data reflect the real-world value?
   - Check: Range validation, regex matching, cross-reference with source.
   - Example: `age` should be between 0 and 150. `email` should match email format.

3. **Consistency:** Is the data consistent across systems and tables?
   - Check: Cross-table reconciliation. `SUM(orders.total) = SUM(invoices.total)`.
   - Example: Customer address in CRM should match address in billing system.

4. **Timeliness:** Is the data up-to-date?
   - Check: `MAX(event_time) >= current_timestamp - INTERVAL 2 HOURS`.
   - Example: Dashboard data should be refreshed within the agreed SLA.

5. **Uniqueness:** Are there no duplicates?
   - Check: `COUNT(*) = COUNT(DISTINCT primary_key)`.
   - Example: No two customers should have the same `customer_id`.

6. **Validity:** Does the data conform to the defined format/type/range?
   - Check: Type checking, enum validation, foreign key constraints.
   - Example: `status` must be one of {'active', 'inactive', 'pending'}.

**Implementation framework:**

| Layer | Checks | When |
|---|---|---|
| **Ingestion** | Schema validation, type checks, null checks | Before loading |
| **Staging** | Uniqueness, range, regex, completeness | After load, before publish |
| **Production** | Consistency, cross-table reconciliation, timeliness | Periodically |
| **Monitoring** | Trend analysis, anomaly detection, alerting | Continuous |

**Tools:** Great Expectations, Deequ (AWS), Soda, custom SQL checks, Delta Lake CHECK constraints.

```sql
-- Example Delta Lake constraint
ALTER TABLE orders ADD CONSTRAINT amount_positive CHECK (amount >= 0);
ALTER TABLE orders ADD CONSTRAINT valid_status CHECK (status IN ('active', 'inactive', 'pending'));
```

### G13: What is incremental loading vs full load? When do you use each?

**Answer:**

**Full Load:**
- The entire dataset is loaded from source to target every run.
- Target table is typically truncated and reloaded (or overwritten via `INSERT OVERWRITE`).
- **Pros:** Simple to implement. No need to track what changed. Guaranteed consistency (target always matches source).
- **Cons:** Slow for large datasets. High resource usage. Can't run frequently (only suitable for daily/weekly schedules).
- **Use case:** Small tables (< 1M rows), dimension tables that fully refresh, or when the source doesn't support incremental extraction.

**Incremental Load:**
- Only new or changed records since the last run are loaded.
- Requires a mechanism to identify what changed:
  - **Watermark column** (`updated_at` timestamp): `WHERE updated_at > last_run_time`.
  - **CDC (Change Data Capture):** Read transaction logs to capture INSERT/UPDATE/DELETE.
  - **Version column** (`version` integer): `WHERE version > last_version`.
- Use `MERGE INTO` (upsert) to update existing records and insert new ones.
- **Pros:** Fast (only processes a small fraction of data). Low resource usage. Can run frequently (hourly, every 15 min).
- **Cons:** More complex to implement. Need to track watermark. Must handle deletes separately. Risk of missing records if watermark is wrong.
- **Use case:** Large fact tables, streaming/near-real-time pipelines, frequently updated data.

**Comparison:**

| Criterion | Full Load | Incremental Load |
|---|---|---|
| **Data volume processed** | Entire dataset | Only changes since last run |
| **Speed** | Slow (proportional to total data) | Fast (proportional to changes) |
| **Complexity** | Low | Medium-High |
| **Frequency** | Daily/weekly | Hourly/continuous |
| **Deletes handling** | Automatic (truncate + reload) | Requires special handling (CDC or soft delete flags) |
| **Idempotency** | Naturally idempotent | Requires MERGE/upsert |
| **Risk** | Low (simple) | Medium (watermark errors, missed records) |

**Best practice:**
- Use incremental loading for large fact tables (orders, events, transactions).
- Use full load for small dimension tables that fully refresh.
- Implement a periodic full-load reconciliation (e.g., weekly) to catch any records missed by incremental loads.
- Store the watermark in a metadata table: `pipeline_metadata(table_name, last_loaded_watermark, last_run_time)`.

### G14: What is the difference between a message queue and pub/sub messaging?

**Answer:**

**Message Queue (Point-to-Point):**
- One producer sends a message to a queue. One consumer receives and processes it.
- Once a consumer acknowledges the message, it's removed from the queue.
- Each message is processed by exactly one consumer.
- Example: AWS SQS, RabbitMQ (queue mode), ActiveMQ.
- Use case: Task distribution, work queues, load balancing across workers.

```
[Producer] --> [Queue] --> [Consumer A]  (or Consumer B, but not both)
```

**Pub/Sub (Publish-Subscribe):**
- One producer (publisher) sends a message to a topic. Multiple consumers (subscribers) each receive a copy.
- Each subscriber independently processes the message.
- Messages are delivered to all active subscribers.
- Example: Kafka (topics), Google Pub/Sub, SNS, NATS.
- Use case: Event notification, fan-out processing, multi-consumer workflows.

```
[Producer] --> [Topic] --> [Consumer A]
                        --> [Consumer B]
                        --> [Consumer C]
```

**Comparison:**

| Criterion | Message Queue | Pub/Sub |
|---|---|---|
| **Consumers per message** | Exactly one | All subscribers |
| **Message persistence** | Removed after ack | Depends on implementation (Kafka retains) |
| **Decoupling** | Consumer-specific | Full decoupling (publisher doesn't know subscribers) |
| **Scaling** | Add more consumers (competing consumers) | Add more subscribers (each gets a copy) |
| **Use case** | Work distribution, task queues | Event broadcasting, multi-consumer processing |

**Kafka (hybrid):**
- Kafka uses topics (pub/sub), but within a consumer group, each partition is consumed by only one consumer (queue-like).
- This provides both pub/sub (multiple consumer groups) and load balancing (within a group).
- Each consumer group independently tracks its offset.

```
[Producer] --> [Kafka Topic] --> [Consumer Group A: partition 0 -> consumer 1, partition 1 -> consumer 2]
                            --> [Consumer Group B: all partitions -> consumer 3]
```

### G15: What is the role of orchestration in data engineering?

**Answer:**

**Orchestration** is the coordination, scheduling, and management of data pipeline execution — ensuring tasks run in the right order, at the right time, with proper error handling and dependency management.

**Key responsibilities of an orchestrator:**

1. **Dependency Management:**
   - Define task dependencies (Task A must complete before Task B).
   - Build DAGs (Directed Acyclic Graphs) of tasks.
   - Support conditional logic (run Task C only if Task A succeeded).

2. **Scheduling:**
   - Time-based: cron schedules (daily at 2 AM, every hour).
   - Event-based: trigger when a file arrives, when a table is updated, or when an upstream pipeline completes.
   - External triggers: API calls or manual runs.

3. **Error Handling and Retries:**
   - Automatic retries with exponential backoff.
   - Dead letter queues for poison-pill messages.
   - Alerting on failures (email, Slack, PagerDuty).

4. **Resource Management:**
   - Allocate compute resources (cluster sizing, autoscaling).
   - Manage concurrency limits (don't run 100 pipelines at once).
   - Pool management for shared resources.

5. **Monitoring and Observability:**
   - Track pipeline status (running, success, failed).
   - Duration metrics, SLA monitoring.
   - Visual dashboards of pipeline health.

**Common Orchestration Tools:**

| Tool | Type | Strengths |
|---|---|---|
| **Apache Airflow** | Open-source | Python-based DAGs, huge ecosystem, flexible |
| **Databricks Workflows (Jobs)** | Managed | Native Databricks integration, serverless, simple |
| **Prefect** | Open-source / Managed | Modern Python API, dynamic DAGs, easy testing |
| **Dagster** | Open-source / Managed | Asset-based, data-aware orchestration, strong lineage |
| **AWS Step Functions** | Managed | AWS-native, visual workflows, serverless |
| **dbt** | SQL-focused | Transforms-only orchestration, SQL-based, testing |

**Best practices:**
- Keep orchestrator logic simple — don't put data processing logic in the orchestrator.
- Make tasks idempotent — safe to retry.
- Use sensors for external dependencies (wait for file arrival).
- Separate compute from orchestration — the orchestrator triggers, the compute processes.
- Monitor and alert on SLA breaches.

### G16: What is data modeling at the conceptual, logical, and physical levels?

**Answer:**

Data modeling is done at three levels of abstraction, from high-level to implementation-specific:

**1. Conceptual Data Model:**
- High-level, business-focused model. Identifies key entities and relationships.
- No attributes, no data types, no technical details.
- Audience: Business stakeholders, product managers.
- Example: "Customers place Orders. Orders contain Products. Products belong to Categories."
- Notation: Entity-Relationship (ER) diagram with entities and relationships only.

**2. Logical Data Model:**
- Adds detail: attributes for each entity, primary keys, foreign keys.
- Normalized (typically 3NF). No consideration of storage or performance.
- Audience: Data architects, data modelers, analysts.
- Example: `Customer(customer_id PK, name, email)`, `Order(order_id PK, customer_id FK, order_date, total_amount)`.
- Still database-agnostic — doesn't specify data types or indexes.

**3. Physical Data Model:**
- Implementation-specific. Includes data types, constraints, indexes, partitioning.
- Denormalized for performance (star/snowflake schema).
- Audience: Database administrators, data engineers.
- Example: `CREATE TABLE dim_customer (customer_sk BIGINT, customer_id STRING, name STRING, ...) PARTITIONED BY (region) CLUSTER BY (customer_id)`.
- Database-specific: different DDL for PostgreSQL vs Snowflake vs Databricks.

**Progression:**
```
Conceptual (What?) --> Logical (How, logically?) --> Physical (How, technically?)
    Entities             Entities + Attributes       Tables + Types + Indexes
    Relationships        Primary/Foreign Keys        Partitioning + Storage
    No attributes        Normalized                  Denormalized for performance
```

**When each is used:**
- Conceptual: Requirements gathering, stakeholder alignment.
- Logical: Data architecture design, normalization, business rules.
- Physical: Implementation, performance optimization, deployment.

### G17: What is the difference between a fact table and a dimension table?

**Answer:**

**Fact Table:**
- Contains quantitative, measurable data (metrics/events) about business processes.
- Each row represents a single event or transaction.
- Columns: foreign keys to dimension tables + numeric measures.
- Typically large (millions to billions of rows).
- Grows continuously over time.
- Example: `fact_sales(order_id, customer_key, product_key, date_key, quantity, amount)`

**Dimension Table:**
- Contains descriptive attributes (context) about the entities in the fact table.
- Each row represents a distinct entity (customer, product, date, location).
- Columns: surrogate key + descriptive attributes.
- Typically small (thousands to millions of rows).
- Changes slowly (SCD).
- Example: `dim_customer(customer_key, customer_id, name, segment, region, signup_date)`

**Comparison:**

| Criterion | Fact Table | Dimension Table |
|---|---|---|
| **Content** | Numeric measures, metrics | Descriptive attributes, context |
| **Size** | Large (billions of rows) | Small (thousands to millions) |
| **Growth** | Fast (new events continuously) | Slow (new entities occasionally) |
| **Keys** | Foreign keys to dimensions | Surrogate key (primary key) |
| **Grain** | One row per event/transaction | One row per entity |
| **Joins** | Center of star schema | Surrounds fact table |
| **Updates** | Append-only (mostly) | SCD (slowly changing) |
| **Example** | `fact_sales`, `fact_page_views` | `dim_customer`, `dim_product`, `dim_date` |

**Types of Fact Tables:**
- **Transaction fact:** One row per transaction event (e.g., each sale).
- **Snapshot fact:** Periodic snapshot (e.g., daily account balance).
- **Accumulating snapshot:** One row per process lifecycle (e.g., order from placed to delivered).

**Types of Dimension Tables:**
- **Conformed dimension:** Shared across multiple fact tables (e.g., `dim_date`, `dim_customer`).
- **Junk dimension:** Combines unrelated low-cardinality flags into one table.
- **Degenerate dimension:** A dimension key stored in the fact table with no separate dimension table (e.g., `order_id`).
- **Role-playing dimension:** Same dimension used multiple times (e.g., `dim_date` as `order_date` and `ship_date`).

### G18: What are the different types of indexes and when do you use them?

**Answer:**

An index is a data structure that improves query speed by enabling fast lookup without scanning the entire table.

**Common Index Types:**

1. **B-Tree Index (default in most RDBMS):**
   - Balanced tree structure. Good for equality and range queries.
   - Use for: `WHERE col = value`, `WHERE col BETWEEN x AND y`, `ORDER BY col`.
   - Example: `CREATE INDEX idx_customer_email ON customers(email)`.

2. **Hash Index:**
   - Hash table structure. Extremely fast for equality lookups.
   - Use for: `WHERE col = value` only. Not for range queries or sorting.
   - Example: `CREATE INDEX idx_user_id ON users USING HASH (user_id)`.

3. **Bitmap Index:**
   - Uses bitmaps. Efficient for low-cardinality columns (few distinct values).
   - Use for: Columns with < 100 distinct values (gender, status, boolean).
   - Not good for: High-cardinality columns or frequently updated tables.

4. **Composite (Multi-Column) Index:**
   - Index on multiple columns. Order matters (leftmost prefix).
   - Use for: `WHERE col1 = x AND col2 = y` queries.
   - Example: `CREATE INDEX idx_name_city ON customers(last_name, city)`.
   - Can be used for `WHERE last_name = 'Smith'` but NOT for `WHERE city = 'NYC'` alone.

5. **Unique Index:**
   - Ensures no duplicate values in the indexed column(s).
   - Use for: Primary keys, email addresses, natural keys.
   - Example: `CREATE UNIQUE INDEX idx_email ON customers(email)`.

6. **Full-Text Index:**
   - Indexes individual words in text columns for text search.
   - Use for: `WHERE CONTAINS(description, 'data engineering')`.
   - Example: `CREATE FULLTEXT INDEX ON articles(content)`.

7. **Clustered Index:**
   - Determines the physical order of rows in the table. One per table.
   - Use for: The most frequently queried/sorted column.
   - Example: Primary key usually gets the clustered index.

**In modern data warehouses (Snowflake, BigQuery, Databricks):**
- Traditional indexes are less relevant — these systems use columnar storage with automatic optimization.
- **Partitioning** replaces clustered indexes for large tables.
- **Clustering keys** (Snowflake) / **Liquid Clustering** (Databricks) replace secondary indexes.
- **Materialized views** pre-compute expensive queries.
- The optimizer handles index selection automatically.

**Best practices:**
- Don't over-index: indexes speed up reads but slow down writes.
- Index columns used in WHERE, JOIN, and ORDER BY clauses.
- Monitor index usage and remove unused indexes.
- For OLAP, prefer partitioning and clustering over traditional indexes.

### G19: What is the difference between a data catalog and a data dictionary?

**Answer:**

**Data Dictionary:**
- A centralized repository of metadata describing the structure of data assets.
- Contains: table names, column names, data types, constraints, descriptions, default values.
- Typically maintained manually or auto-generated from the database schema.
- Scope: Usually limited to a single database or system.
- Audience: Developers, DBAs, data engineers.
- Example: A spreadsheet or document listing all tables and columns with descriptions.

**Data Catalog:**
- A broader, more sophisticated metadata management system that goes beyond structural metadata.
- Contains: data dictionary info PLUS business metadata (owners, stewards, usage), technical metadata (lineage, profiles, quality scores), operational metadata (pipeline runs, freshness).
- Supports search, discovery, and collaboration.
- Scope: Enterprise-wide, spanning multiple databases, data lakes, warehouses, and pipelines.
- Audience: Data engineers, analysts, data scientists, business users.
- Examples: Unity Catalog (Databricks), DataHub, Amundsen, Collibra, Alation, AWS Glue Data Catalog.

**Comparison:**

| Criterion | Data Dictionary | Data Catalog |
|---|---|---|
| **Scope** | Single database/system | Enterprise-wide |
| **Metadata type** | Structural (schema, types) | Structural + business + technical + operational |
| **Search/Discovery** | Limited or manual | Full-text search, tagging, recommendations |
| **Lineage** | No | Yes (table and column-level) |
| **Access control** | No | Yes (RBAC, column masking) |
| **Quality metrics** | No | Yes (DQ scores, freshness, completeness) |
| **Collaboration** | Limited | Comments, ratings, requests |
| **Automation** | Manual or schema-generated | Automated (connectors, crawlers, listeners) |
| **Audience** | Technical (developers, DBAs) | Technical + business users |

**Evolution:** The data catalog is the modern evolution of the data dictionary. As organizations grew from single databases to complex data ecosystems, simple data dictionaries became insufficient. Data catalogs add lineage, governance, discovery, and collaboration on top of structural metadata.

**In Databricks:** Unity Catalog serves as both the data dictionary (schema, types, columns) and data catalog (lineage, tags, access control, search, quality) for all data assets in the workspace.

### G20: What is the difference between eventual consistency and strong consistency?

**Answer:**

**Strong Consistency (Linearizability):**
- After a write completes, any subsequent read (from any node) returns the latest value.
- All nodes agree on the same state at all times.
- There is no lag between write and read visibility.
- **Guarantee:** You always read the most recent write.
- **Trade-off:** Higher latency (must coordinate across nodes), lower availability during partitions.
- **Use case:** Financial transactions, inventory management, anything where reading stale data causes errors.
- **Examples:** Traditional RDBMS (single node), Google Spanner, MongoDB (with majority read concern).

**Eventual Consistency:**
- After a write, reads may return stale data for some period. Eventually (when replication catches up), all nodes converge to the same value.
- There is a lag (replication delay) between write and when all nodes see it.
- **Guarantee:** If no new writes occur, all nodes will eventually return the last written value.
- **Trade-off:** Lower latency, higher availability (no coordination needed).
- **Use case:** Social media feeds, product recommendations, caching, counters where temporary inconsistency is acceptable.
- **Examples:** Cassandra (default), DynamoDB (eventual consistency mode), S3 (read-after-write within a region, cross-region eventual).

**Comparison:**

| Criterion | Strong Consistency | Eventual Consistency |
|---|---|---|
| **Read freshness** | Always latest | May be stale temporarily |
| **Write latency** | Higher (coordinates across nodes) | Lower (no coordination) |
| **Availability** | Lower (may block during partition) | Higher (accepts writes regardless) |
| **Complexity** | Higher (consensus protocols) | Lower (async replication) |
| **Use case** | Financial, transactional | Social, analytics, caching |

**Levels between strong and eventual (tunable consistency):**
- **Read-your-writes:** A client always sees their own writes.
- **Session consistency:** Within a session, reads see all previous writes in that session.
- **Monotonic reads:** Once a client reads a value, they never see an older value.
- **Bounded staleness:** Data is at most K seconds old.

**In data engineering:**
- Data warehouses (Snowflake, BigQuery, Databricks) provide **strong consistency** — analytics requires accurate data.
- Data lakes (S3 without Delta) provide **eventual consistency** for cross-region replication — but within a single region, S3 provides read-after-write consistency.
- Delta Lake on S3 provides **strong consistency** through its transaction log — the log ensures readers always see a consistent snapshot.
- Streaming systems (Kafka) provide **strong consistency** within a partition (ordered), but **eventual consistency** across consumers (consumer lag).

## 📦 Data Modeling & Schema Design

### Q1: Real-time click events at 1M events/minute — real-time dashboards + daily batch reports

**Scenario:** You have a streaming source of user click events arriving at 1M events/minute. The business needs both real-time dashboards (last 15 min) and daily batch reports. How would you design the pipeline and storage layers?

**Answer:**

**Architecture Overview (Lambda-style or Kappa-style):**

1. **Ingestion Layer:**
   - Events land in **Kafka** (or Kinesis) with multiple partitions (e.g., 50-100) to handle 1M/min throughput.
   - Retain data in Kafka for at least 24-48 hours to allow reprocessing if needed.

2. **Real-Time Path (Speed Layer):**
   - A **Spark Structured Streaming** or **Flink** job consumes from Kafka.
   - Maintain rolling 15-minute window aggregates in an in-memory store (Redis) or a low-latency OLAP store (e.g., Apache Pinot, ClickHouse).
   - The dashboard queries this store directly for sub-second response.
   - Use **watermarks** (e.g., 2 minutes) to handle late-arriving events.

3. **Batch Path (Batch Layer):**
   - Kafka events are also written to S3 (or ADLS) in Parquet format via a micro-batch writer (e.g., Kafka Connect S3 sink or Spark Structured Streaming `writeStream` to S3).
   - Partition by `event_date/hour` for efficient querying.
   - A nightly **Spark batch job** processes the full day's data and writes to a Delta/Iceberg table.
   - This batch table serves the daily reports with full accuracy and complex aggregations.

4. **Serving Layer (Reconciliation):**
   - The batch layer's output overwrites/reconciles the speed layer's approximate results.
   - For real-time dashboards, the speed layer is "good enough" — the batch layer provides the source of truth.

**Key Trade-offs:**
- Lambda = two code paths (stream + batch), maintenance overhead.
- Kappa = single streaming path, but harder for complex historical reprocessing.
- If using **Delta Lake**, you can use `MERGE` to reconcile speed + batch results in one table.

**Storage Choices:**
- Raw events → S3/ADLS (Parquet, partitioned by date)
- Real-time aggregates → Redis or ClickHouse/Pinot
- Batch aggregates → Delta/Iceberg table in the warehouse

**Why this works:** The speed layer handles latency-sensitive dashboards (~seconds), while the batch layer ensures accuracy and supports complex ad-hoc queries on historical data.

### Q2: E-commerce dimensional model with returns, exchanges, and partial refunds

**Scenario:** You need to build a dimensional model for an e-commerce platform with orders, products, customers, and sellers. Sales can be returned, exchanged, or partially refunded. How do you model the fact and dimension tables?

**Answer:**

**Core Tables:**

1. **`dim_customer`** (Type 1 SCD): customer_id, name, email, signup_date, segment
2. **`dim_product`** (Type 2 SCD): product_id (surrogate), natural_product_id, name, category, price (track price changes over time)
3. **`dim_seller`** (Type 1 SCD): seller_id, name, rating, region
4. **`dim_date`**: date_id, full_date, year, quarter, month, day_of_week, is_holiday

**Fact Tables:**

5. **`fact_sales`** (Grain = one row per order line item):
   - `sales_key` (surrogate), `order_id`, `customer_key`, `product_key`, `seller_key`, `date_key`
   - `quantity`, `unit_price`, `gross_amount`, `discount_amount`, `net_amount`
   - `order_status` (placed, shipped, delivered, returned, exchanged, partially_refunded)

6. **`fact_returns`** (Grain = one row per return/refund event):
   - `return_key`, `original_sales_key` (FK to fact_sales), `return_type` (full_return, partial_refund, exchange)
   - `return_date_key`, `refund_amount`, `exchange_product_key` (nullable)
   - `return_reason`, `processed_by`

**Handling Each Scenario:**

- **Full Return:** A row in `fact_returns` with `return_type = 'full_return'`, `refund_amount = original net_amount`. The corresponding `fact_sales.order_status` is updated to `returned`.

- **Partial Refund:** A row in `fact_returns` with `return_type = 'partial_refund'`, `refund_amount < original net_amount`. The `fact_sales.order_status` becomes `partially_refunded`.

- **Exchange:** A row in `fact_returns` with `return_type = 'exchange'`, `exchange_product_key` populated. A new row is inserted into `fact_sales` for the exchanged product (linked via `order_id` or a new `exchange_order_id`).

**Net Revenue Calculation:**
```sql
SELECT 
    d.month,
    SUM(f.net_amount) AS gross_revenue,
    SUM(COALESCE(r.refund_amount, 0)) AS total_refunds,
    SUM(f.net_amount) - SUM(COALESCE(r.refund_amount, 0)) AS net_revenue
FROM fact_sales f
LEFT JOIN fact_returns r ON f.sales_key = r.original_sales_key
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.month
```

**Why separate fact_returns?**
- Returns happen on a different date than the original sale — you need to attribute refunds to the correct period.
- Avoids polluting `fact_sales` with refund events at a different grain.
- Allows analytics like "return rate by product, seller, time period."

**SCD Type 2 on `dim_product`:** If product price changes, you need historical accuracy. Each price change creates a new surrogate key, and `fact_sales` references the key that was valid at the time of sale.

### Q3: GPS pings from 10,000 delivery vehicles — partitioning and clustering strategy

**Scenario:** You're designing a table to store GPS pings from a fleet of 10,000 delivery vehicles. Each ping has timestamp, lat, lon, speed, and vehicle_id. What partitioning and clustering strategy would you use, and why?

**Answer:**

**Data Volume Estimate:**
- 10,000 vehicles × 1 ping/10 seconds = 1,000 pings/sec = ~86M pings/day = ~31B pings/year
- At ~50 bytes/ping (compressed Parquet), that's ~1.5TB/day.

**Partitioning Strategy: `event_date`**
- Partition by `event_date` (daily partitions).
- Daily partitions keep each partition manageable (~86M rows, ~1.5TB compressed).
- Most queries filter by date range ("show me pings for vehicle X on Oct 5").
- Avoid over-partitioning (hourly would create 8,760 partitions/year — too many small files, metastore overhead).

**Clustering/Sorting Strategy: `vehicle_id`**
- Cluster (or sort within partitions) by `vehicle_id`.
- This enables fast queries like "all pings for vehicle V123 on a given date" — the engine skips data blocks that don't contain that vehicle_id.
- Also beneficial for joins with a vehicle dimension table (co-partitioned lookups).

**In Databricks (Delta Lake with Liquid Clustering):**
```sql
CREATE TABLE fleet_gps (
    event_timestamp TIMESTAMP,
    vehicle_id STRING,
    lat DOUBLE,
    lon DOUBLE,
    speed DOUBLE
)
USING DELTA
PARTITIONED BY (event_date DATE)  -- derived from event_timestamp
CLUSTER BY (vehicle_id);
```

**Why not partition by vehicle_id?**
- 10,000 vehicles → 10,000 partitions × 365 days = 3.65M partitions — far too many.
- Partition pruning by date + clustering by vehicle_id gives the best of both worlds.

**Additional Optimizations:**
- **Z-ORDER** (if not using Liquid Clustering) on `vehicle_id` to co-locate data for the same vehicle.
- **OPTIMIZE** regularly to compact small files and maintain read performance.
- **Retention policy:** Consider tiered storage — hot (last 30 days on SSD), cold (older on S3) — or time-travel with `VACUUM` after 90 days.
- **Downsampling:** For historical analytics, create aggregated tables (e.g., one ping per minute per vehicle) to reduce query costs on large date ranges.

## 🔄 ETL/ELT Pipeline Design

### Q4: 50 source tables with mixed sizes and staggered arrival — orchestration design

**Scenario:** You have 50 source tables that need to be loaded daily into your warehouse. 5 of them are huge (10B+ rows), 10 are medium, and the rest are small. Some sources arrive at different times during the day. How do you orchestrate this?

**Answer:**

**1. Categorize and Prioritize:**
- **Huge tables (5):** Incremental load (CDC or watermark-based), not full refresh. Use parallel Spark jobs with appropriate partitioning.
- **Medium tables (10):** Incremental or full refresh depending on size and change frequency.
- **Small tables (35):** Full refresh — simple, fast, low risk.

**2. Orchestration with Airflow (or Databricks Workflows):**

```
DAG: daily_pipeline_load
  ├── [Group 1] Small tables (35) → parallel tasks (batch of 10 at a time)
  ├── [Group 2] Medium tables (10) → parallel tasks (batch of 3 at a time)
  ├── [Group 3] Huge tables (5) → sequential or limited parallelism (2 at a time)
  ├── [Dependency] All groups complete → trigger downstream transformations
  └── [Notification] Alert on failure, retry logic
```

**3. Handling Staggered Arrival:**
- Use **file arrival sensors** (Airflow `FileSensor` or S3 event notifications) instead of fixed schedules.
- Each table has its own task with a sensor that waits for the source file/data to arrive.
- Tables that arrive early start processing immediately; others wait.
- Set a SLA timeout — if data doesn't arrive by a cutoff, alert the on-call engineer.

**4. Incremental Loading for Huge Tables:**
- Use a **watermark column** (e.g., `updated_at`) to load only new/changed rows since last run.
- Track last-loaded watermark in a metadata table: `pipeline_metadata(table_name, last_loaded_watermark, last_run_time)`.
- For CDC: use Delta Lake's Change Data Feed or Debezium for MySQL/PostgreSQL.
- Merge into the target Delta table using `MERGE INTO`.

**5. Resource Management:**
- Use different cluster sizes: small jobs on a small cluster, huge table loads on a large autoscaling cluster.
- In Databricks: use job clusters (not interactive) with autoscaling (e.g., 2–16 workers).
- Use **pools** to reduce cluster spin-up time.

**6. Idempotency and Recovery:**
- Each task is idempotent — re-running it produces the same result (MERGE with watermark).
- If a task fails, Airflow retries 3× with exponential backoff.
- Partial failures don't block other independent tables.
- A `backfill` script can re-run any table for a specific date range.

**7. Monitoring:**
- Track row counts, load duration, and watermark lag per table.
- Alert if row count drops >20% vs. rolling 7-day average (possible data loss).
- Dashboard showing pipeline health across all 50 tables.

### Q5: Pipeline that went from 2 hours to 6 hours — debugging approach

**Scenario:** A daily pipeline that used to finish in 2 hours now takes 6 hours. Walk me through your debugging approach step by step.

**Answer:**

**Step 1: Isolate the Bottleneck**
- Check pipeline run history — was the degradation sudden (overnight) or gradual (over weeks)?
  - Sudden → likely a code change, data anomaly, or infrastructure issue.
  - Gradual → likely data growth, file fragmentation, or resource contention.
- Break the pipeline into stages and check each stage's duration. Identify which stage(s) slowed down.

**Step 2: Check Data Volume**
- Compare today's input data volume vs. normal. Did a source table suddenly 5× in size?
- Check for data skew — one partition or key may have an abnormally large share of records.
- Look for duplicate data — upstream issue may have sent duplicates.

**Step 3: Check Spark UI / Query Profile**
- **Spark UI:** Look at the Stages tab — which stage is taking the most time?
  - Long `shuffle read/write` → data skew or bad partitioning.
  - High `GC time` → memory pressure, need more executors or optimize data structures.
  - Spill to disk → not enough memory, need more executor memory or fewer concurrent tasks.
  - Task skew → one task takes 30 min while others take 2 min → skewed key.
- **Query Profile (Databricks):** Check scanned bytes, shuffle bytes, number of files scanned.

**Step 4: Common Causes and Fixes**

| Symptom | Likely Cause | Fix |
|---|---|---|
| Sudden slowdown | Code change deployed | Compare git diff, revert if needed |
| Gradual slowdown | Data growth | Add partitioning, optimize table, increase cluster size |
| Shuffle heavy | Join explosion or skew | Broadcast join for small tables, salting for skewed keys |
| Many small files | No OPTIMIZE run | Run `OPTIMIZE` to compact files |
| GC pressure | Too much data in memory | Increase executor memory, reduce batch size, use columnar format |
| Stale stats | CBO making bad plan | Run `ANALYZE TABLE` to refresh statistics |
| Contention | Concurrent jobs on same cluster | Separate clusters or schedule offsets |

**Step 5: Quick Wins Checklist**
- [ ] Was there a recent code deploy? → Check CI/CD history.
- [ ] Is `OPTIMIZE` overdue? → Run it.
- [ ] Are there many small files? → Compact.
- [ ] Is a join causing a shuffle? → Try broadcast join.
- [ ] Is data skewed? → Salt the skewed key.
- [ ] Is the cluster too small? → Scale up / enable autoscaling.
- [ ] Are there upstream issues? → Check source data volume and quality.

**Step 6: Long-term Prevention**
- Set up alerts on pipeline duration (e.g., >2.5 hours → warning, >4 hours → critical).
- Add performance regression tests — track key metrics over time.
- Document baseline performance for each stage.

### Q6: Duplicates and schema changes from source — handling both

**Scenario:** You have a pipeline where the source system sometimes sends duplicate records and occasionally has schema changes (new columns, renamed columns). How do you handle both?

**Answer:**

**Part A: Handling Duplicates**

1. **Deduplication at ingestion:**
   - If the source has a unique key (e.g., `event_id`), use `DROP Duplicates` in Spark:
     ```python
     df_deduped = df.dropDuplicates(['event_id'])
     ```
   - If using Delta Lake, use `MERGE INTO` to upsert — naturally handles duplicates by key.

2. **Idempotent writes:**
   - Use `MERGE INTO` (upsert) instead of `INSERT INTO` — if the same record arrives twice, the second one updates the first rather than creating a duplicate.

3. **Watermark + dedup window (streaming):**
   - In Structured Streaming, use `dropDuplicatesWithinWatermark` to dedup within a time window.
   ```python
   df = df.withWatermark('event_time', '10 minutes').dropDuplicatesWithinWatermark(['event_id'])
   ```

4. **Quality checks:**
   - After loading, run a count check: `SELECT event_id, COUNT(*) FROM table GROUP BY event_id HAVING COUNT(*) > 1`.
   - Alert if duplicates are found post-load.

**Part B: Handling Schema Changes**

1. **Schema Evolution in Delta Lake:**
   - Delta supports automatic schema evolution with `mergeSchema = true`:
     ```python
     df.write.option('mergeSchema', 'true').format('delta').mode('append').save(path)
     ```
   - New columns are added automatically; existing columns are preserved.

2. **Column Renames:**
   - This is trickier — a rename looks like a deleted column + a new column.
   - Maintain a **schema mapping table** that maps source column names to target column names.
   - Apply the mapping as a transformation step before writing.
   ```python
   column_mapping = {'old_name': 'new_name', 'cust_id': 'customer_id'}
   for old_col, new_col in column_mapping.items():
       if old_col in df.columns:
           df = df.withColumnRenamed(old_col, new_col)
   ```

3. **Schema Registry:**
   - Use a schema registry (e.g., Confluent Schema Registry for Kafka) to manage and version schemas.
   - Detect schema changes at the source, alert the team, and plan migrations.

4. **Schema Validation:**
   - Before loading, validate the incoming schema against expected schema.
   - If schema changes are detected, quarantine the data and alert — don't silently load incompatible data.
   ```python
   expected_columns = {'id', 'name', 'email', 'amount'}
   incoming_columns = set(df.columns)
   new_columns = incoming_columns - expected_columns
   missing_columns = expected_columns - incoming_columns
   if new_columns or missing_columns:
       # Alert and handle
   ```

5. **Column Pruning and Nullable Handling:**
   - If a source column is removed, the target table column stays but receives nulls for new data.
   - If a new column appears, decide: add it to the target (schema evolution) or drop it initially and add in a planned migration.

**Best Practice:** Combine both — use MERGE for dedup + `mergeSchema` for schema evolution + schema validation alerts for unexpected changes.

### Q7: Dashboard numbers don't match source system — investigation

**Scenario:** A business team says the numbers in the dashboard don't match the source system. Walk me through your investigation approach.

**Answer:**

**Step 1: Reproduce the Discrepancy**
- Get specific details: which dashboard, which metric, which time period, what's the expected vs. actual value?
- Pull the exact numbers from both the dashboard and the source system for the same time period.
- Determine the magnitude: is it a small % difference (filtering issue) or a large one (missing data or pipeline failure)?

**Step 2: Check Time Window Alignment**
- Is the dashboard showing data for Oct 1–7 while the source system shows Oct 1–7 in a different timezone?
- Check if the dashboard uses UTC while the source uses local time (or vice versa).
- Verify date boundaries — does "daily" mean calendar day or rolling 24 hours?

**Step 3: Check Filters and Transformations**
- Does the dashboard apply filters the source query doesn't? (e.g., excluding cancelled orders, filtering by status).
- Are there transformations (aggregations, joins, deduplication) that change the count?
- Check the exact SQL/logic behind the dashboard metric.

**Step 4: Check Pipeline Completeness**
- Did the pipeline run successfully for the time period in question?
- Were there any partial failures, skipped partitions, or retried runs?
- Check the pipeline run logs and data freshness timestamp.
- Compare row counts: `SELECT COUNT(*) FROM warehouse_table WHERE date = '2026-10-05'` vs. source.

**Step 5: Check for Data Loss or Duplication**
- Are records missing in the warehouse that exist in the source? (ingestion gap)
- Are there duplicates inflating warehouse counts? (dedup failure)
- Compare at the record level: pull 10 specific records from both systems and compare.

**Step 6: Check for Late-Arriving Data**
- The pipeline may have run before all source data arrived. If the pipeline runs at 2 AM but some records arrive at 3 AM, they'll be missing from yesterday's load.
- Check if a re-run or backfill is needed.

**Step 7: Check Joins and Fan-Out**
- If the dashboard query joins multiple tables, a one-to-many join could inflate counts.
- Verify join keys are unique on the dimension side.

**Step 8: Document and Prevent**
- Once the root cause is found, document it.
- Add data quality checks (reconciliation alerts) comparing source counts to warehouse counts.
- Implement automated reconciliation: `source_count vs. warehouse_count` with alerting on delta > threshold.
- Add lineage tracking so future discrepancies can be traced through the pipeline.

**Communication:** Keep the stakeholder informed throughout — acknowledge the issue, share your investigation plan, provide a timeline, and deliver the root cause + fix.

## 📊 Data Quality & Reliability

### Q8: Financial reporting pipeline — small discrepancy + prevention

**Scenario:** You have a critical pipeline feeding financial reporting. A downstream consumer reports that yesterday's numbers are wrong by a small margin. How do you investigate, and what safeguards would you add to prevent this in the future?

**Answer:**

**Investigation Phase:**

1. **Quantify the discrepancy:**
   - Get the exact expected vs. actual values. "Small margin" could be $100 or $100K.
   - Pull the source-of-truth numbers and compare with the warehouse output for the same period.

2. **Check for rounding/truncation:**
   - Are amounts stored as `FLOAT` instead of `DECIMAL`? Floating-point precision can cause small discrepancies.
   - Verify data types: financial amounts should always be `DECIMAL(p, s)`, never `FLOAT` or `DOUBLE`.

3. **Check for missing records:**
   - Were some transactions excluded by a filter? (e.g., transactions with `status = 'pending'` excluded from reporting).
   - Compare source record count vs. warehouse record count for the period.

4. **Check for late-arriving data:**
   - Did some transactions arrive after the pipeline ran? Check the watermark/max timestamp in the source vs. what was loaded.

5. **Check for currency conversion:**
   - If multi-currency, was the exchange rate applied correctly? Was the rate from the right date?

6. **Check transformations:**
   - Trace the lineage: source → staging → final table. Compare intermediate results at each step.
   - Look for `COALESCE`, `NULLIF`, `CASE` logic that might transform values unexpectedly.

**Safeguards to Add:**

| Safeguard | Description |
|---|---|
| **Reconciliation checks** | Automated comparison: `SUM(source.amount) vs. SUM(warehouse.amount)` with alert on delta > $1 |
| **Data type enforcement** | Use `DECIMAL(18, 2)` for all monetary fields; reject `FLOAT` |
| **Completeness check** | `source_count = warehouse_count` assertion post-load |
| **Freshness SLA** | Alert if data is older than expected (e.g., max timestamp < current time - 2 hours) |
| **Schema enforcement** | Use Delta's `CONSTRAINT` clauses: `ALTER TABLE ... ADD CONSTRAINT amount_positive CHECK (amount >= 0)` |
| **Audit trail** | Log every pipeline run with: input rows, output rows, rejected rows, duration, checksum |
| **Dual validation** | Two independent queries compute the same metric; alert if they disagree |
| **Lineage tracking** | Full column-level lineage so any future discrepancy can be traced end-to-end |

**For financial pipelines specifically:**
- Implement **double-entry validation**: debits must equal credits per transaction.
- Use **idempotent MERGE** operations so re-runs don't double-count.
- Store a **checksum/hash** per batch and validate on load.
- Implement **data contracts** with upstream teams — enforce schema, expected volume, and SLA agreements.

### Q9: Third-party API resilience — partial data, timeouts, schema changes

**Scenario:** Your pipeline ingests data from a third-party API. Sometimes the API returns partial data, sometimes it times out, and sometimes the schema changes. How do you build resilience?

**Answer:**

**1. Retry with Exponential Backoff:**
```python
import time
import requests

@retry(max_attempts=5, backoff_base=2, backoff_max=60)
def fetch_from_api(url, params):
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()
```
- Start with 2s wait, then 4s, 8s, 16s, 32s (capped at 60s).
- Use jitter (random delay) to avoid thundering herd if multiple clients retry simultaneously.

**2. Handle Partial Data:**
- Compare today's response with historical patterns:
  - Row count within 80–120% of rolling 7-day average → proceed.
  - Row count outside range → quarantine + alert + don't load.
- Check for null rates per field — if a critical field is >10% null, flag it.
- Implement a **completeness score**: `(received_records / expected_records) * 100`.

**3. Circuit Breaker:**
- If the API fails 5 consecutive times, stop hitting it for 5 minutes.
- Prevents cascading failures and protects the API from being overwhelmed.
```python
circuit_open = False
failure_count = 0
FAILURE_THRESHOLD = 5
COOLDOWN_SECONDS = 300

def call_api(url):
    global circuit_open, failure_count
    if circuit_open:
        raise Exception('Circuit breaker open — API unavailable')
    try:
        result = fetch_from_api(url)
        failure_count = 0
        return result
    except Exception:
        failure_count += 1
        if failure_count >= FAILURE_THRESHOLD:
            circuit_open = True
            schedule_circuit_reset(COOLDOWN_SECONDS)
        raise
```

**4. Schema Change Handling:**
- Store the expected schema as a JSON contract file.
- On each API response, validate the schema:
  - New fields → log warning, continue (additive change is safe).
  - Missing required fields → quarantine data, alert, don't load.
  - Type changes → quarantine, alert, review.
- Use a **schema version** field from the API (if available) to detect breaking changes.

**5. Fallback / Degraded Mode:**
- If the API is down, use the last successful load's data with a "stale data" flag.
- Alert downstream consumers that data may be stale.
- Never silently serve stale data without marking it.

**6. Idempotent Loading:**
- Use `MERGE INTO` so retries don't create duplicates.
- Track which batches have been loaded: `load_metadata(batch_id, api_url, status, record_count, loaded_at)`.

**7. Dead Letter Queue:**
- Records that fail validation (bad schema, nulls in required fields) go to a dead letter table.
- These can be reviewed, corrected, and reprocessed later.

**8. Monitoring:**
- API latency p50/p99 tracking.
- Success rate dashboard.
- Alert on: timeout rate > 5%, partial data rate > 10%, schema mismatch detected.

### Q10: Multi-dimensional data quality framework design

**Scenario:** How would you design a data quality framework that validates data across multiple dimensions (completeness, accuracy, consistency, timeliness, uniqueness) without significantly impacting pipeline runtime?

**Answer:**

**Framework Architecture:**

```
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│  Data Load   │───▶│  DQ Check    │───▶│  Pass?       │───▶│  Publish     │
│  (Staging)   │    │  Layer       │    │  Decision     │    │  to Prod     │
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
                           │                                     ▲
                           ▼                                     │
                    ┌──────────────┐                        ┌─────┴──────┐
                    │  DQ Results  │                        │  Quarantine │
                    │  Store       │                        │  Table       │
                    └──────────────┘                        └────────────┘
```

**The Five DQ Dimensions:**

| Dimension | Check Type | Example |
|---|---|---|
| **Completeness** | Null checks, row count | `SELECT COUNT(*) WHERE critical_field IS NULL` — should be 0 |
| **Accuracy** | Range checks, regex | `amount BETWEEN 0 AND 1000000`, `email REGEXP '^[^@]+@[^@]+\\.[^@]+$'` |
| **Consistency** | Cross-table checks | `SUM(orders.total) = SUM(invoices.total)` for same period |
| **Timeliness** | Freshness check | `MAX(event_time) >= current_timestamp - INTERVAL 2 HOURS` |
| **Uniqueness** | Duplicate check | `COUNT(*) = COUNT(DISTINCT id)` — no duplicates |

**Implementation Strategy (Minimizing Runtime Impact):**

1. **Push checks to the source:**
   - Run DQ checks on a **sample** (e.g., 1% of data) for expensive checks like cross-table consistency.
   - Use **Delta Lake constraints** for cheap, automatic checks:
   ```sql
   ALTER TABLE orders ADD CONSTRAINT 
     amount_positive CHECK (amount >= 0);
   ALTER TABLE orders ADD CONSTRAINT 
     unique_order_id CHECK (order_id IS NOT NULL);
   ```

2. **Layered checks:**
   - **Layer 1 (Fast, pre-load):** Schema validation, null checks, type checks — run on incoming data before loading.
   - **Layer 2 (Medium, post-load):** Uniqueness, range checks, completeness — run on the loaded staging table.
   - **Layer 3 (Slow, async):** Cross-table consistency, lineage validation, statistical drift detection — run async, don't block the pipeline.

3. **Use Great Expectations or custom framework:**
```python
# Example with custom DQ checks
import pandas as pd

class DataQualityCheck:
    def __init__(self, name, check_fn, severity='error'):
        self.name = name
        self.check_fn = check_fn
        self.severity = severity  # 'error' = block, 'warning' = alert only
    
    def run(self, df):
        result = self.check_fn(df)
        return {'check': self.name, 'passed': result, 'severity': self.severity}

# Define checks
checks = [
    DataQualityCheck('no_null_ids', lambda df: df['id'].isnull().sum() == 0, 'error'),
    DataQualityCheck('amount_positive', lambda df: (df['amount'] >= 0).all(), 'error'),
    DataQualityCheck('row_count_range', lambda df: 1000 < len(df) < 10000000, 'warning'),
]

# Run all checks
results = [check.run(df) for check in checks]
blocking_failures = [r for r in results if not r['passed'] and r['severity'] == 'error']
if blocking_failures:
    raise DataQualityError(f'Blocking checks failed: {blocking_failures}')
```

4. **DQ Results Store:**
   - Every check result is stored: `dq_results(table_name, check_name, passed, severity, value, expected, run_date)`.
   - Enables trending: "completeness of `users.email` has been declining over the last 30 days."

5. **Action on Failure:**
   - **Error-level:** Block pipeline, quarantine data, alert on-call.
   - **Warning-level:** Log, alert, but allow pipeline to proceed.
   - **Metrics-level:** Record for dashboarding, no alerting.

6. **Performance Tips:**
   - Run checks as Spark SQL on the staging table (not row-by-row Python).
   - Use `CHECK` constraints in Delta for automatic enforcement at write time.
   - Parallelize independent checks.
   - Cache the staging table for checks.

## ⚡ Performance & Optimization

### Q11: Spark job with severe skew in join stage — diagnosis and fix

**Scenario:** A Spark job that processes 5TB of data runs daily. You notice severe skew in one of the join stages. How do you diagnose and fix it?

**Answer:**

**Diagnosis:**

1. **Identify the skewed stage:**
   - Open Spark UI → Stages tab → sort by duration.
   - Look for a stage where one task takes much longer than others (e.g., Task 1 takes 45 min, Tasks 2–200 take 2 min each).
   - Check the **Shuffle Read** column — skewed tasks will have disproportionately large shuffle reads.

2. **Identify the skewed key:**
   - In Spark UI → SQL/DataFrame tab → find the join node → check the join keys.
   - Or run a diagnostic query:
   ```python
   df.groupBy('join_key').count().orderBy('count', ascending=False).show(20)
   # If one key has 500M rows and others have 10K, that's your skewed key.
   ```

3. **Determine the cause:**
   - Common cause: a **"null" or default key** that many records map to.
   - Or a **dominant entity** (e.g., one customer_id represents 80% of all orders).
   - Or a **one-to-many explosion** — joining a fact table with a bridge table.

**Fixes (in order of preference):**

**Fix 1: Broadcast Join (if one side is small)**
```python
# If the skewed table is small (< 10MB by default, can increase to ~8GB)
result = large_df.join(broadcast(small_df), 'key')
```
- Eliminates shuffle entirely — no skew possible.
- Works when one side of the join fits in executor memory.

**Fix 2: Salt the Skewed Key (Salting Technique)**
```python
from pyspark.sql.functions import col, concat, lit, rand, explode, array

# Identify the hot key
hot_key = 'CUSTOMER_001'

# Add salt to the large table's hot key
large_salted = large_df.withColumn(
    'salted_key',
    when(col('join_key') == hot_key,
         concat(col('join_key'), lit('_'), (rand() * 9).cast('int')))
    .otherwise(col('join_key'))
)

# Replicate the small table's hot key 10 times
small_exploded = small_df.withColumn(
    'salted_key',
    when(col('join_key') == hot_key,
         explode(array([concat(col('join_key'), lit(f'_{i}')) for i in range(10)])))
    .otherwise(col('join_key'))
)

# Join on salted key — distributes the hot key across 10 partitions
result = large_salted.join(small_exploded, 'salted_key')
```
- The hot key `CUSTOMER_001` becomes `CUSTOMER_001_0` through `CUSTOMER_001_9`.
- Each salted variant goes to a different partition, distributing the load.

**Fix 3: Adaptive Query Execution (AQE) — Spark 3.x**
```python
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
```
- AQE automatically detects skew at runtime and splits skewed partitions.
- This is the easiest fix — enable it and let Spark handle it.
- Works best when the skew is moderate.

**Fix 4: Pre-Aggregate Before Join**
- If you're joining a large fact table with a dimension and then aggregating:
```python
# Instead of: fact.join(dim).groupBy().agg()
# Do: fact.groupBy().agg() first, then join with dim
fact_agg = fact.groupBy('dim_key', 'date').agg(sum('amount').alias('total'))
result = fact_agg.join(dim, 'dim_key')  # Much smaller, less skew
```

**Fix 5: Filter or Split the Hot Key**
- Separate the hot key, process it independently, and union the results:
```python
hot_data = df.filter(col('key') == hot_key)
cold_data = df.filter(col('key') != hot_key)
# Process hot_data with more parallelism/repartition
cold_result = cold_data.join(dim, 'key')
hot_result = hot_data.repartition(100, 'key').join(dim, 'key')
result = cold_result.union(hot_result)
```

**Best Practice:** Always enable AQE first, then apply salting or splitting if AQE isn't sufficient.

### Q12: 1B-row fact table JOIN 100-row dimension table — optimization

**Scenario:** You have a query that joins a 1B-row fact table with a 100-row dimension table and it's very slow. The dimension table rarely changes. How do you optimize this?

**Answer:**

**Root Cause:** A shuffle join is happening because Spark doesn't realize the dimension table is small enough to broadcast. Or the broadcast threshold is too low.

**Solution 1: Broadcast Join (Primary Fix)**
```python
# Explicit broadcast hint
result = fact_df.join(broadcast(dim_df), 'dim_key')
```
Or in SQL:
```sql
SELECT /*+ BROADCAST(dim) */ *
FROM fact
JOIN dim ON fact.dim_key = dim.dim_key
```
- The dimension table (100 rows, likely < 1KB) is sent to all executors.
- No shuffle of the 1B-row fact table — massive performance win.
- Expected: from minutes/hours to seconds.

**Solution 2: Increase Broadcast Threshold**
```python
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '100MB')  # Default is 10MB
```
- If the dimension table is slightly larger than the default threshold, increase it.
- Be careful — broadcasting too large a table causes OOM.

**Solution 3: Pre-Join / Denormalize (if dim rarely changes)**
- Since the dimension rarely changes, pre-join at load time:
```sql
-- Create a pre-joined table
CREATE TABLE fact_enriched AS
SELECT f.*, d.category, d.region, d.segment
FROM fact f
JOIN dim d ON f.dim_key = d.dim_key
```
- Eliminates the join at query time entirely.
- Best for dashboards that always need the same dimension columns.
- Trade-off: storage cost increases, but query latency drops dramatically.

**Solution 4: Map-Side Join (for file-based pipelines)**
- In Spark, use a broadcast variable to distribute the dimension to all nodes:
```python
dim_map = dim_df.collectAsMap()
dim_broadcast = spark.sparkContext.broadcast(dim_map)

# Use in UDF or mapPartitions
def enrich_partition(rows):
    dim = dim_broadcast.value
    for row in rows:
        yield {**row, 'category': dim.get(row['dim_key'], {}).get('category', 'Unknown')}

result = fact_rdd.mapPartitions(enrich_partition)
```

**Solution 5: Cache the Dimension Table**
```python
dim_df.cache()  # or persist to memory
# Spark will keep it in memory for subsequent joins
```
- Useful if the same dimension is joined multiple times in a pipeline.

**Summary:** The primary fix is almost always **broadcast join**. A 100-row table is trivially small. If the query is still slow after broadcast, check for other issues (file format, partition pruning, table statistics).

### Q13: Dashboard query from 10s to 5min after 3x data growth — diagnosis and solutions

**Scenario:** A dashboard query that previously ran in 10 seconds is now taking 5 minutes. The data volume has grown 3x. What are all the things you'd check and what solutions might you apply?

**Answer:**

**Phase 1: Diagnose**

1. **Check the query profile / execution plan:**
   - What's the query doing? Full scan? Join? Aggregation? Subquery?
   - Is it scanning more data than necessary? Check `bytes scanned`.
   - Are there any shuffle operations?

2. **Check partition pruning:**
   - Is the query filtering on the partition column? If not, it's doing a full table scan.
   - Example: table is partitioned by `event_date`, but query filters on `created_at` — partition pruning doesn't apply.
   - Verify the filter predicate matches the partition column.

3. **Check file health:**
   - Are there many small files? Run `DESCRIBE DETAIL table` — check `numFiles`.
   - Small files kill scan performance — lots of file open/close overhead.
   - When was the last `OPTIMIZE` run? Run `OPTIMIZE table` to compact.

4. **Check table statistics:**
   - Run `ANALYZE TABLE table COMPUTE STATISTICS` — stale stats cause the optimizer to pick bad plans.
   - Check if the Cost-Based Optimizer (CBO) is enabled.

5. **Check caching:**
   - Was the query previously benefiting from a cached result that has since expired?
   - Is the dashboard query cached in the BI tool? Check cache settings.

6. **Check concurrency:**
   - Are there other queries running simultaneously? Resource contention can slow things down.

**Phase 2: Solutions**

| Problem | Solution |
|---|---|
| **Full table scan** | Add partition filter; ensure query uses partition column |
| **No partition pruning** | Repartition table by commonly-filtered column |
| **Many small files** | Run `OPTIMIZE` / `COMPACT` to merge small files |
| **Expensive join** | Pre-join into a materialized view; use broadcast join for small tables |
| **Stale statistics** | Run `ANALYZE TABLE` to refresh stats |
| **Complex aggregation** | Pre-aggregate into a summary table; refresh nightly |
| **No clustering** | Add Z-ORDER or Liquid Clustering on filter columns |
| **Large data scan** | Use columnar format (Parquet/Delta) with column pruning |
| **Repeated computation** | Cache the result; use a materialized view |

**Specific Optimizations:**

1. **Create a pre-aggregated summary table:**
```sql
CREATE TABLE daily_sales_summary AS
SELECT 
    event_date, 
    product_category,
    SUM(amount) as total_sales,
    COUNT(*) as order_count
FROM fact_sales
GROUP BY event_date, product_category
-- Refresh daily via scheduled job
```
- Dashboard queries the summary table (millions of rows) instead of the fact table (billions).

2. **Materialized View (if supported):**
```sql
CREATE MATERIALIZED VIEW sales_mv
REFRESH ON SCHEDULE EVERY 1 HOUR
AS SELECT ... FROM fact_sales GROUP BY ...;
```

3. **Liquid Clustering / Z-ORDER:**
```sql
OPTIMIZE table ZORDER BY (product_category, region)
```
- Co-locates data for frequently-filtered columns → less data scanned per query.

4. **Column Pruning:**
- Ensure the query only selects needed columns, not `SELECT *`.

5. **Increase warehouse/cluster size:**
- If the query is I/O bound, a larger cluster may not help — focus on data layout.
- If the query is compute-bound (complex aggregations), more compute power helps.

6. **Result caching:**
- Enable result caching in Databricks SQL — repeated queries return cached results.

**Rule of thumb:** 3x data growth shouldn't cause 30x query slowdown. If it does, the query was already inefficient — it was just small enough to get away with it. Optimization is now mandatory.

## 🌊 Streaming & Real-Time

### Q14: Real-time fraud detection — sub-5-second flagging

**Scenario:** You need to build a real-time fraud detection pipeline. Events come from Kafka, and you need to flag suspicious transactions within 5 seconds. How do you architect this?

**Answer:**

**Architecture Overview:**

```
[Transaction Events]
        │
        ▼
   [Kafka] ──────────────────────▶ [Stream Processing (Flink/Spark SS)]
        │                                    │
        │                                    ├──▶ [Rule Engine] ──▶ Flag → [Alert System]
        │                                    ├──▶ [ML Model Scoring] ──▶ Score → [Alert System]
        │                                    └──▶ [State Store] (rolling windows, user history)
        │
        └──▶ [S3/ADLS] (raw events, for batch retraining)
```

**Key Components:**

1. **Ingestion — Kafka:**
   - Transaction events produced to Kafka topic `transactions`.
   - Partition by `user_id` or `merchant_id` to ensure ordering per key.
   - Retain 7 days for replay capability.

2. **Stream Processing — Flink (preferred for low latency) or Spark Structured Streaming:**

   **A. Rule-Based Detection (Fast, deterministic):**
   ```python
   # Flink/Spark: Check velocity — >5 transactions in 1 minute per user
   from pyspark.sql.functions import *

   suspicious = transactions \
       .withWatermark('event_time', '10 seconds') \
       .groupBy(
           window('event_time', '1 minute'),
           'user_id'
       ) \
       .agg(count('*').alias('txn_count')) \
       .filter('txn_count > 5')
   ```

   **B. ML-Based Scoring (More sophisticated):**
   - Pre-trained model deployed as a REST endpoint (Model Serving).
   - Stream processor calls the endpoint for each transaction (or micro-batch).
   - Model returns a fraud probability score (0–1).
   - If score > threshold (e.g., 0.85), flag as suspicious.
   ```python
   # Score micro-batches against model endpoint
   def score_batch(batch_df):
       # Call MLflow model serving endpoint
       scores = call_model_endpoint(batch_df)
       return batch_df.withColumn('fraud_score', scores)
   
   flagged = transactions \
       .transform(score_batch) \
       .filter('fraud_score > 0.85')
   ```

   **C. Stateful Checks:**
   - Maintain rolling state per user: last 10 transactions, average amount, geo locations.
   - Flag anomalies: transaction in a new country, amount 10x above average, velocity spike.
   - Use Flink keyed state or Spark stateful operations.

3. **State Management:**
   - **Flink:** RocksDB state backend (handles TB-scale state, checkpointed).
   - **Spark Structured Streaming:** Use `flatMapGroupsWithState` for custom stateful logic.
   - Store user profiles in an external KV store (Redis/DynamoDB) for fast lookups.

4. **Latency Budget (5 seconds):**
   - Kafka → Processing: < 1 second (consumer lag monitoring).
   - Rule checks: < 1 second (in-memory computation).
   - ML scoring: < 2 seconds (batch inference, not per-record, to amortize overhead).
   - Alert dispatch: < 1 second (push to alert topic/webhook).

5. **Alerting & Action:**
   - Flagged transactions → Kafka topic `fraud_alerts`.
   - Consumer service reads alerts and takes action: block transaction, send SMS, create review ticket.
   - Dashboard: real-time count of flagged transactions, false positive rate.

6. **Handling False Positives:**
   - Log all flagged transactions with features and scores for offline analysis.
   - Feedback loop: analyst labels transactions as fraud/not-fraud → retrains model weekly.

7. **Fault Tolerance:**
   - Exactly-once processing with Kafka transactions and checkpointing.
   - If the processing engine restarts, it resumes from the last checkpoint.
   - Idempotent alert writing (dedup by transaction_id).

### Q15: Kafka consumer falling behind by 2 hours — immediate and long-term fixes

**Scenario:** You're processing a Kafka stream and the consumer group falls behind by 2 hours during peak traffic. How do you handle this — both immediately and as a long-term fix?

**Answer:**

**Immediate Actions (Stop the Bleeding):**

1. **Scale up the consumer group:**
   - Add more consumer instances to the group. Kafka will rebalance partitions across consumers.
   - Maximum parallelism = number of partitions (can't have more consumers than partitions).
   ```bash
   # If running on K8s: scale the deployment
   kubectl scale deployment stream-processor --replicas=20
   ```

2. **Increase consumer resources:**
   - If consumers are CPU-bound: increase CPU/memory per consumer.
   - If consumers are I/O-bound: increase batch size, reduce processing overhead.

3. **Increase batch size (process more per poll):**
   ```python
   # Kafka consumer config
   consumer_configs = {
       'max.poll.records': 5000,       # Default 500 → increase
       'fetch.max.bytes': 52428800,    # 50MB per fetch
       'max.partition.fetch.bytes': 10485760  # 10MB per partition
   }
   ```

4. **Skip non-critical processing temporarily:**
   - If the pipeline does enrichment + ML scoring + validation, temporarily skip ML scoring.
   - Process raw events quickly to catch up, then backfill enrichment later.

5. **Parallel replay (if catching up on backlog):**
   - Spin up a temporary consumer group with more partitions to replay the backlog.
   - Process the 2-hour backlog in parallel, then switch back to the primary consumer.

**Long-Term Fixes:**

1. **Increase Kafka partitions:**
   - If you have 10 partitions and 10 consumers, you can't scale further.
   - Increase to 50 or 100 partitions → allows 50–100 parallel consumers.
   - Note: increasing partitions requires careful planning (key ordering is preserved within partition, but data needs redistribution).

2. **Optimize processing logic:**
   - Profile the consumer: what's the bottleneck? Deserialization? Transformation? Output writes?
   - **Batch writes** instead of row-by-row: accumulate 5 seconds of events, then write as a batch.
   - **Async processing**: if calling external APIs, use async I/O instead of blocking calls.
   - **Avoid expensive operations** in the hot path: move heavy enrichment to batch.

3. **Auto-scaling:**
   - Set up auto-scaling based on consumer lag metric:
   ```python
   # Monitor consumer lag
   lag = kafka_consumer_lag(topic='events', group='processor')
   if lag > 10000:  # More than 10K messages behind
       scale_up_consumer_group()
   ```
   - Use Kubernetes HPA or Databricks autoscaling.

4. **Handle peak traffic with buffer:**
   - Accept that peak traffic may exceed real-time capacity.
   - Design for eventual consistency — process peak backlog within 30 minutes of peak ending.
   - Use a **DLQ (Dead Letter Queue)** for poison-pill messages that cause processing failures and back up the queue.

5. **Pre-process at ingestion:**
   - If possible, filter or pre-aggregate at the Kafka producer level to reduce message volume.
   - Use Kafka Streams for lightweight pre-processing before the heavy consumer.

6. **Partition by throughput, not by key:**
   - If one key (e.g., a hot customer) generates 50% of traffic, that partition is always the bottleneck.
   - Consider a different partitioning strategy that distributes load more evenly.

### Q16: Stream join with SCD — point-in-time correctness

**Scenario:** You need to join a real-time stream of orders with a slowly-changing dimension table of customers. How do you ensure point-in-time correctness?

**Answer:**

**The Problem:**
- Customer attributes change over time (e.g., customer moves from "Silver" to "Gold" tier on Oct 5).
- An order from Oct 3 should be joined with the customer's "Silver" attributes, not the current "Gold" attributes.
- Using a simple lookup on the current dimension table would produce wrong results for historical orders.

**Solution 1: SCD Type 2 + Temporal Join (Preferred)**

1. **Store the dimension as SCD Type 2:**
```sql
CREATE TABLE dim_customer_scd2 (
    customer_sk BIGINT,        -- Surrogate key (unique per version)
    customer_id STRING,       -- Natural key
    tier STRING,
    region STRING,
    valid_from TIMESTAMP,
    valid_to TIMESTAMP,       -- NULL or '9999-12-31' for current version
    is_current BOOLEAN
)
```

2. **Temporal join (point-in-time lookup):**
```sql
-- For each order, join with the customer version that was valid at order_time
SELECT 
    o.order_id,
    o.order_time,
    o.amount,
    c.tier,
    c.region
FROM orders_stream o
JOIN dim_customer_scd2 c
    ON o.customer_id = c.customer_id
    AND o.order_time >= c.valid_from
    AND (o.order_time < c.valid_to OR c.valid_to IS NULL)
```

This ensures each order gets the customer attributes that were valid at the time of the order.

**Solution 2: Change Data Capture (CDC) Stream + As-of Join**

If the dimension changes are also available as a CDC stream (e.g., via Debezium):

```python
# In Flink: use a temporal table join
# Register the customer CDC stream as a temporal table
customer_changes.createTemporalTableFunction('customer_id', 'change_time')

# Join orders with customer changes as-of the order time
result = orders.join_lateral(
    customer_changes('order_time', 'customer_id')
)
```

**Solution 3: Spark Structured Streaming + Stateful Lookup**

For Spark Structured Streaming, use a combination of stream-static join with periodic dimension refresh:

```python
# Load the latest version of each customer
# For new orders, lookup the current customer attributes
# For backfill/historical, use the SCD2 temporal join above

# In streaming context:
customers_current = spark.read.format('delta').load('/dim_customer') \
    .filter('is_current = true')

# Stream-static join (uses current dimension state)
enriched_stream = orders_stream.join(customers_current, 'customer_id')
```

**Caveat:** Stream-static joins in Spark use the dimension state at query start. To refresh, restart the streaming query or use `streamingJoins` with a refreshable source.

**Solution 4: Dual-Write with Versioned State Store**

For true real-time enrichment:

1. Maintain a **versioned KV store** (Redis, RocksDB):
   - Key: `customer_id`, Value: `{tier, region, valid_from, valid_to, version}`
   - Updated by CDC events from the dimension source.

2. Stream processor looks up the KV store for each order:
```python
def enrich_order(order, kv_store):
    customer = kv_store.get(order['customer_id'])
    # If customer changed since the order was placed, use the version valid at order_time
    if order['order_time'] < customer['valid_from']:
        customer = kv_store.get_historical(order['customer_id'], order['order_time'])
    return {**order, 'tier': customer['tier'], 'region': customer['region']}
```

**Key Principle:** Point-in-time correctness requires knowing **which version of the dimension was valid when the fact occurred**. SCD Type 2 + temporal join is the industry standard approach.

## 🏗️ Architecture & System Design

### Q17: Ad impression system — 500K events/sec, sub-second latency (Meta/Google style)

**Scenario:** Design a system to ingest, process, and serve ad impression data at 500K events/second with sub-second latency for a real-time bidding dashboard.

**Answer:**

**Full Architecture:**

```
[Ad Impressions (500K/sec)]
        |
        v
[Kafka - 100+ partitions, replication factor 3]
        |
        +--> [Stream Processor (Flink) - real-time aggregation]
        |          |
        |          +--> [Redis / Aerospike - sub-second lookups]
        |          +--> [ClickHouse / Apache Pinot - real-time OLAP]
        |                    |
        |                    +--> [Dashboard (sub-second queries)]
        |
        +--> [S3 Sink (Parquet) - raw event storage]
        |          |
        |          +--> [Spark Batch - nightly aggregation -> Delta/Iceberg]
        |
        +--> [Flink - fraud/quality filtering -> filtered topic]
```

**1. Ingestion Layer (500K events/sec):**
- Kafka with 100-200 partitions, replication factor 3.
- Partition by `ad_id` or `publisher_id` to distribute load.
- Producer: batch size 50-100 events, linger time 5ms, compression LZ4.
- Expected throughput: 500K/sec / 100 partitions = 5K events/sec/partition.

**2. Real-Time Processing (Flink preferred for sub-second latency):**
- Flink with keyed state and incremental aggregates.
- Aggregate per-minute windows: impressions, clicks, CTR, spend.
- State backend: RocksDB (handles TB-scale state, checkpointed to S3).
- Checkpoint interval: 10 seconds.

**3. Real-Time Serving Store:**
- Apache Pinot or ClickHouse for OLAP queries on fresh data.
- Pinot: ingests from Kafka in real-time, supports sub-second SQL queries.
- Dashboard queries: `SELECT ad_id, SUM(impressions), SUM(clicks) FROM realtime_metrics WHERE ts >= now() - 15 minutes GROUP BY ad_id`.

**4. Batch Layer (for historical analytics):**
- Kafka to S3 (Parquet, partitioned by date/hour).
- Nightly Spark job: aggregate full-day data to Delta table.
- Historical dashboards query the Delta table; real-time dashboards query Pinot/ClickHouse.

**5. Sub-Second Latency Budget:**

| Component | Latency |
|---|---|
| Producer -> Kafka | 10-50ms |
| Kafka -> Flink consumer | 10-50ms |
| Flink processing + aggregation | 50-200ms |
| Flink -> Pinot/ClickHouse | 50-100ms |
| Pinot query -> Dashboard | 100-300ms |
| **Total** | **~500ms** |

**6. Reliability:**
- Kafka replication factor 3 - survives single broker failure.
- Flink checkpoints to S3 every 10s - automatic recovery on failure.
- Pinot/ClickHouse replicas - at least 2.
- Alerting: consumer lag, error rate, latency p99.

**7. Data Volume Estimate:**
- 500K/sec x 50 bytes/event = 25 MB/sec = ~2 TB/day raw.
- Raw storage: 2 TB/day x 365 = 730 TB/year -> tiered storage (hot 7 days, warm 90 days, cold 1 year).

### Q18: Centralized metadata, lineage, and monitoring for hundreds of pipelines

**Scenario:** You have hundreds of data pipelines across multiple teams. How would you design a centralized metadata, lineage, and monitoring system?

**Answer:**

**Architecture Overview:**

```
+-------------------------------------------------------------------+
|                    Centralized Data Platform                       |
|                                                                   |
|  +----------+  +----------+  +----------+  +----------+          |
|  | Pipeline |  | Metadata |  | Lineage  |  | Monitoring|          |
|  | Registry|  |  Catalog |  | Tracker  |  |  Dashboard|          |
|  +----------+  +----------+  +----------+  +----------+          |
|       |              |              |              |               |
|       +--------------+--------------+--------------+               |
|                          |                                        |
|                    +-----+-----+                                   |
|                    | Metadata  |                                   |
|                    |  Store    |                                   |
|                    | (Postgres |                                   |
|                    | / Atlas)  |                                   |
|                    +-----------+                                   |
+-------------------------------------------------------------------+
```

**1. Pipeline Registry (Cataloging):**
- Every pipeline is registered with metadata: name, owner, team, schedule, SLA, source tables, target tables.
- Use a tool like DataHub, Amundsen, or Unity Catalog (Databricks native).
- Store in a metadata database (Postgres or the catalog's native store).

**2. Lineage Tracking:**
- Automatically capture column-level lineage from pipeline code.
- In Spark: use `SparkListener` or Delta Lake's built-in lineage.
- In SQL: parse the query plan to extract source -> target column mappings.
- Use OpenLineage (open standard) or Databricks Unity Catalog lineage.
- Benefits: impact analysis ("if raw.orders changes, what downstream pipelines are affected?"), root cause analysis, compliance.

**3. Monitoring and Observability:**

| Metric | Source | Alert Threshold |
|---|---|---|
| **Pipeline success/failure** | Orchestrator (Airflow/Databricks) | Failure -> PagerDuty |
| **Duration** | Orchestrator | > 2x baseline -> warning |
| **Data freshness** | Max timestamp in target | > SLA -> critical |
| **Row count** | Post-load count | < 50% of 7-day avg -> critical |
| **Data quality** | DQ checks | Error-level check fails -> block |
| **Schema drift** | Schema comparison | Change detected -> alert owner |
| **Consumer feedback** | Dashboard ratings | Rating < 3/5 -> investigate |

**4. Dashboard (Single Pane of Glass):**
- All pipelines with status (green/yellow/red).
- Freshness by table (how stale is each table?).
- Lineage graph (interactive - click a table, see upstream/downstream).
- SLA compliance (% of pipelines meeting SLA this week).
- Data quality scores per table.
- Tools: Grafana, custom React app, or data observability platforms (Monte Carlo, Databand).

**5. Governance:**
- Ownership: Every table and pipeline has a documented owner.
- Data contracts: Upstream teams commit to schema, volume, and freshness SLAs.
- Deprecation policy: Unused tables are identified and cleaned up quarterly.
- Access control: Unity Catalog or IAM policies - least privilege.

**6. Self-Service Discovery:**
- Engineers can search for tables, see lineage, owners, freshness, and quality without asking around.
- Reduces tribal knowledge dependency and onboarding time.

### Q19: 500+ tables — access control, governance, PII, and GDPR at scale

**Scenario:** Your company has 500+ tables in the data warehouse. How do you manage access control, data governance, PII detection, and compliance (GDPR) at scale?

**Answer:**

**1. Access Control (Unity Catalog):**

- **Catalog -> Schema -> Table** hierarchy with inheritance.
- **Role-based access control (RBAC):**
  - `data_engineer` role: read/write to `engineering` schema.
  - `analyst` role: read-only to `analytics` schema.
  - `pii_authorized` role: read access to PII columns.
- **Column-level masking:** for PII columns, mask based on role:
```sql
-- Unity Catalog column mask
CREATE MASKING FUNCTION mask_email AS (val STRING)
RETURN CASE
  WHEN is_member('pii_authorized') THEN val
  ELSE CONCAT('***@', split(val, '@')[1])
END;

ALTER TABLE customers ALTER COLUMN email SET MASK mask_email;
```
- **Row-level security (RLS):** filter rows based on user attributes:
```sql
-- Only see your region's data
CREATE FUNCTION region_filter AS (region STRING)
RETURN CASE
  WHEN is_member('admin') THEN TRUE
  ELSE region = current_user_region()
END;

ALTER TABLE sales ALTER COLUMN region SET ROW FILTER region_filter;
```

**2. PII Detection and Classification:**

- **Automated PII scanning:**
  - Use Databricks' built-in classification or a tool like Collibra/Immuta.
  - Scan table schemas and sample data to identify PII columns:
    - Regex: SSN, email, phone.
    - Column names: `ssn`, `email`, `phone`, `address`, `dob`, `credit_card`.
  - Tag discovered PII columns automatically with governance tags.

```sql
-- Tag PII columns in Unity Catalog
ALTER TABLE customers COLUMN email SET TAG pii = 'true';
ALTER TABLE customers COLUMN ssn SET TAG pii = 'true', pii_type = 'ssn';
```

- **Classification tiers:**
  - **Public:** No restriction.
  - **Internal:** Company-wide access.
  - **Confidential:** Need-to-know basis.
  - **Restricted:** PII, financial - strict access, audit logging.

**3. GDPR Compliance:**

- **Right to Erasure (Right to be Forgotten):**
  - Implement a deletion pipeline: given a user_id, find all tables containing their data and delete/anonymize.
  - Use lineage to identify all downstream tables derived from PII sources.
  - For Delta tables: `DELETE FROM table WHERE user_id = 'target_user'`.
  - For immutable storage (S3): overwrite with anonymized version.

- **Data minimization:**
  - Only collect and store PII that's strictly necessary.
  - Anonymize where possible (hash user IDs, generalize ZIP codes).

- **Consent management:**
  - Track consent per user per data use case.
  - Filter out users who haven't consented to analytics processing.

- **Audit trail:**
  - Log every access to PII: who, when, what, why.
  - `system.access.audit` table in Databricks for audit logging.
  - Retain audit logs for compliance-required duration (e.g., 7 years).

- **Data retention policies:**
  - Define retention periods per data type (e.g., transaction data: 7 years, web logs: 90 days).
  - Automated deletion/purging: `VACUUM` Delta tables, TTL on S3 objects.

**4. Governance Workflow:**

| Component | Implementation |
|---|---|
| **Data catalog** | Unity Catalog - searchable, with descriptions, owners, tags |
| **Access requests** | Ticketing system (Jira) -> approval -> automated grant |
| **Regular access review** | Quarterly review of who has access to restricted data |
| **Data stewardship** | Assign a data steward per domain (finance, HR, product) |
| **Policy enforcement** | Automated checks: "no PII in non-restricted schemas" |
| **Compliance reporting** | Generate reports on: PII locations, access logs, retention |

**5. Scalability:**
- Use automation: scripts/infrastructure-as-code to apply tags and policies to new tables.
- Policy as code: Define governance policies in YAML/JSON, apply via CI/CD.
- Periodic audits: Automated scans to find untagged PII, over-granted access.
- Self-service: Users request access through a portal; approvals are workflowed.

### Q20: Legacy warehouse to lakehouse migration — 2000+ tables, zero downtime

**Scenario:** You need to migrate a legacy data warehouse (e.g., Teradata/Oracle) to a modern lakehouse. 2000+ tables, 500+ downstream reports, and zero downtime requirement. How do you plan and execute this migration?

**Answer:**

**Phased Migration Strategy:**

**Phase 0: Discovery and Assessment (Weeks 1-4)**
- Inventory all tables: name, size, row count, update frequency, dependencies, downstream consumers.
- Prioritize tables:
  - Tier 1 (Critical): Financial, operational - migrate first with extensive testing.
  - Tier 2 (Important): Analytics, reporting - migrate second.
  - Tier 3 (Low priority): Historical, rarely used - migrate last or archive.
- Identify dependencies: which tables feed which reports? Use lineage if available, or parse ETL code.
- Identify complex transformations: stored procedures, views, UDFs that need rewriting.

**Phase 1: Foundation Setup (Weeks 3-6, overlaps Phase 0)**
- Set up the lakehouse: Databricks workspace, Unity Catalog, storage (S3/ADLS).
- Define naming conventions, partitioning strategies, and table formats (Delta).
- Set up CI/CD for table creation scripts.
- Build the migration framework: scripts to extract from legacy, transform, load to lakehouse.

**Phase 2: Dual-Run Strategy (Weeks 5-16)**

For each batch of tables (e.g., 50-100 tables per batch):

1. **Migrate the table:**
   - Extract from legacy (bulk for historical, CDC for ongoing).
   - Transform schema if needed (e.g., Teradata types -> Delta types).
   - Load to lakehouse as Delta table.

2. **Dual-run ETL:**
   - Run both legacy and new ETL pipelines in parallel.
   - Compare outputs row-by-row for each table:
   ```sql
   -- Reconciliation query
   SELECT 
     'row_count' AS metric,
     (SELECT COUNT(*) FROM legacy.table) AS legacy_val,
     (SELECT COUNT(*) FROM lakehouse.table) AS lakehouse_val,
     ABS(legacy_val - lakehouse_val) AS diff;
   -- Also: SUM of key columns, checksum comparison, sample row comparison.
   ```

3. **Dual-run reports:**
   - Point reports to both legacy and new tables.
   - Compare report outputs.
   - Flag discrepancies and fix.

4. **Gradual cutover:**
   - Once a table passes validation for N consecutive days:
     - Switch downstream reports to read from the lakehouse.
     - Keep the legacy table running (shadow mode) for 2 weeks as safety net.
     - After 2 weeks with no issues, decommission the legacy table.

**Phase 3: Decommission (Weeks 16-20)**
- Once all tables are migrated and reports are cut over:
  - Stop legacy ETL pipelines.
  - Archive legacy data (if needed for compliance).
  - Decommission legacy warehouse (cost savings!).

**Key Strategies for Zero Downtime:**

| Strategy | Description |
|---|---|
| **Dual-run** | Both systems run in parallel; cutover is gradual, not big-bang |
| **Reverse replication** | Optionally replicate lakehouse changes back to legacy as safety |
| **Feature flags** | Reports can switch between legacy/new with a config flag |
| **Shadow traffic** | Send production queries to both systems, compare results |
| **Canary cutover** | Switch 5% of reports to new, verify, then 25%, 50%, 100% |

**Handling Complex Logic:**
- Stored procedures: Rewrite as Spark notebooks or SQL UDFs.
- Views: Recreate as views in Unity Catalog or materialized views.
- UDFs: Rewrite in Python (Spark UDF) or SQL functions.
- Type mapping:
  | Legacy Type | Delta Type |
  |---|---|
  | Teradata `VARCHAR` | `STRING` |
  | Teradata `INTEGER` | `INT` |
  | Teradata `TIMESTAMP` | `TIMESTAMP` |
  | Oracle `NUMBER(p,s)` | `DECIMAL(p,s)` |
  | Oracle `CLOB` | `STRING` |

**Risk Mitigation:**
- Each batch migration has a rollback plan - if issues arise, revert to legacy.
- Validate 100% of Tier 1 tables; sample-validate Tier 2/3.
- Maintain a migration dashboard: tables migrated, validated, cut over, decommissioned.
- Weekly stakeholder reviews to report progress and surface blockers.

**Timeline Estimate:**
- 2000+ tables over ~4-5 months with a team of 5-8 engineers.
- Critical path: dependency analysis and dual-run validation.

## 🧩 Behavioral / Scenario

### Q21: Stakeholder needs a new metric for Monday morning — 3 unknown tables

**Scenario:** A stakeholder comes to you on Friday afternoon saying they need a new metric for a Monday morning executive review. The metric requires joining 3 tables you've never worked with. How do you approach this?

**Answer:**

**Hour 1: Understand the Requirement**
- Sit down with the stakeholder and ask:
  - What exactly is the metric? Get the business definition, not just the name.
  - What granularity? (e.g., daily revenue by region, or weekly active users by segment?)
  - What time period? (last 30 days? QTD? YTD?)
  - Who is the audience? (C-level needs simplicity; VP of Sales needs detail.)
  - Are there existing reports that compute something similar? If yes, use them as a baseline.

**Hour 2: Explore the Tables**
- Find the 3 tables in the catalog/data dictionary.
- Read the schema: columns, types, comments, sample data.
  ```sql
  DESCRIBE TABLE catalog.schema.table_name;
  SELECT * FROM catalog.schema.table_name LIMIT 10;
  ```
- Identify join keys: which columns connect the tables?
- Check for existing lineage or documentation.
- Look for existing notebooks/queries that use these tables (regexAssetSearch or catalog search).

**Hour 3: Prototype the Query**
- Write the SQL query in a notebook.
- Start simple: join the 3 tables and compute the metric.
- Run on a small sample (1 day or LIMIT 1000) to validate quickly.
- Check for: duplicate rows from joins, nulls in key columns, unexpected values.
- Refine the query.

**Hour 4: Validate with Stakeholder**
- Show preliminary results to the stakeholder.
- Ask: does this number make sense? Is it in the right ballpark?
- Compare with any known benchmarks or existing reports.

**Hour 5-6: Productionize (Minimal Viable)**
- Write the final query as a SQL view or a notebook.
- If time permits: add basic formatting (display as a table or simple chart).
- If the stakeholder needs it in a specific tool (Tableau, Looker, dashboard), export or connect.

**Hour 7-8: Documentation & Handoff**
- Document:
  - The metric definition (business logic).
  - The SQL query and which tables it uses.
  - Any assumptions made (e.g., excluded certain statuses, used a specific timezone).
  - Known limitations (e.g., "data is only as fresh as last night's pipeline run").
- Communicate assumptions to the stakeholder — transparency builds trust.

**Monday Morning:**
- Be available for questions during the executive review.
- If the metric gets challenged, explain the methodology confidently.

**Post-Meeting Follow-up:**
- If the metric is useful, productionize it properly:
  - Add it as a scheduled pipeline.
  - Add data quality checks.
  - Add to the catalog with proper documentation.
  - Set up automated refresh.

**Key Principles:**
- Prioritize delivering value over perfection. A "good enough" number on time beats a perfect number late.
- Document assumptions — you may not have time to validate everything.
- Communicate risk: "This is a quick calculation; I recommend validating against source data next week."

### Q22: Pipeline silently dropping 2% of records for 3 months — communication and fix

**Scenario:** You discover that a production pipeline has been silently dropping 2% of records for the past 3 months. The data feeds a machine learning model. How do you communicate this, fix the pipeline, and handle the downstream impact?

**Answer:**

**Step 1: Assess the Impact (Before Communicating)**
- Determine which records are being dropped: is it random or systematic?
  - Random 2%: less severe, but still a data quality issue.
  - Systematic (e.g., all records from a specific source, region, or with a specific attribute): much worse — the dataset is biased.
- Identify the root cause:
  - A filter that's too aggressive?
  - A join that drops unmatched rows (should be a LEFT JOIN instead of INNER JOIN)?
  - A parsing error that rejects certain formats?
  - A schema mismatch causing silent failures?
- Quantify: how many records total? 2% of what volume?

**Step 2: Communicate (Severity-Based)**

**If systematic bias in the ML training data:**
- This is a **high-severity issue**. Notify immediately:
  - Data team lead.
  - ML team / model owner.
  - Business stakeholder who relies on the model's output.
- Use clear language: "We discovered that 2% of records (approximately X records) have been dropped from the pipeline since [date]. The dropped records appear to be [systematic/random]. This affects the ML model's training data."

**If random and low impact:**
- Still notify, but with lower urgency. Include in the weekly status update.

**Communication Template:**
```
Subject: [Action Required] Data Quality Issue - Pipeline X dropping 2% of records

Summary: The [pipeline_name] pipeline has been dropping ~2% of records since [date].
Root cause: [explain briefly]
Impact: [which downstream consumers/models/reports are affected]
Magnitude: [X records over Y months]
Plan: [fix timeline]
Next steps: [what the ML/business team needs to do]
```

**Step 3: Fix the Pipeline**
- Fix the root cause (e.g., change INNER JOIN to LEFT JOIN, fix the filter, handle the parsing error).
- Add a data quality check to catch this in the future:
```sql
-- Post-load check: compare input count to output count
SELECT 
  'record_drop_check' AS check_name,
  input_count,
  output_count,
  (input_count - output_count) AS dropped,
  ROUND((input_count - output_count) * 100.0 / input_count, 2) AS drop_pct,
  CASE WHEN (input_count - output_count) * 100.0 / input_count > 1.0 THEN 'FAIL' ELSE 'PASS' END AS status
FROM (SELECT COUNT(*) AS input_count FROM source_table),
     (SELECT COUNT(*) AS output_count FROM target_table);
```

**Step 4: Backfill the Missing Data**
- Re-run the pipeline for the affected 3-month period to recover the dropped records.
- Merge the recovered records into the target table.
- Validate that the backfill is complete and correct.

**Step 5: Assess ML Model Impact**
- Work with the ML team to:
  - Determine if the model was trained on biased data.
  - Re-train the model with the corrected dataset.
  - Compare model performance before and after the fix.
  - If the model is in production, evaluate whether predictions have been systematically wrong.
  - Consider rolling back to a previous model version if the current one is unreliable.

**Step 6: Prevent Future Occurrences**
- Add automated row count reconciliation (input vs. output) as a pipeline check.
- Add alerting: if drop rate > 0.5%, alert the on-call engineer.
- Add a dashboard showing data completeness trends.
- Conduct a post-mortem and document lessons learned.
- Review other pipelines for similar issues.

**Key Principle:** When you find a data quality issue, **over-communicate**. Stakeholders would rather know about a problem and decide how to handle it than discover it later and realize you knew.

### Q23: Two teams building similar pipelines on the same source — resolution

**Scenario:** Two teams in your company are independently building similar pipelines that consume the same source data with different transformations. How do you resolve this?

**Answer:**

**Step 1: Discover and Understand**
- Map out both pipelines: source tables, transformations, outputs, consumers, schedules.
- Identify what's the same and what's different:
  - Same: both read `raw.orders`, both aggregate by day.
  - Different: Team A filters by `status = 'completed'`, Team B includes all statuses. Team A computes `SUM(amount)`, Team B computes `COUNT(DISTINCT customer_id)`.
- Are the outputs reconcilable? Can one pipeline serve both teams' needs?

**Step 2: Bring Teams Together**
- Organize a meeting with both team leads.
- Present the findings: both pipelines are reading the same source, with overlapping but not identical logic.
- Frame it as a collaboration opportunity, not a blame game.
- Questions to ask:
  - Can we agree on a shared base layer (common transformations)?
  - Can each team build their team-specific logic on top of the shared layer?
  - Are there business reasons the logic differs? (If yes, both may be correct.)

**Step 3: Propose a Layered Architecture**

```
[raw.orders]
      |
      v
[shared.silver_orders]  -- Common cleaning, dedup, type casting
      |
      +--> [team_a.gold_orders_summary]  -- Team A's specific aggregations
      |
      +--> [team_b.gold_customer_activity]  -- Team B's specific aggregations
```

- **Shared Silver Layer:** Common transformations that both teams agree on:
  - Data cleaning (null handling, type casting).
  - Deduplication.
  - Standardized column names.
  - Both teams own and maintain this layer together.

- **Team-Specific Gold Layers:** Each team builds their specific logic on top:
  ```sql
  -- Team A's gold layer
  CREATE TABLE team_a.gold_orders_summary AS
  SELECT date, SUM(amount) AS total_revenue
  FROM shared.silver_orders
  WHERE status = 'completed'
  GROUP BY date;

  -- Team B's gold layer
  CREATE TABLE team_b.gold_customer_activity AS
  SELECT date, COUNT(DISTINCT customer_id) AS active_customers
  FROM shared.silver_orders
  GROUP BY date;
  ```

**Step 4: Establish Governance**
- Define ownership: who maintains the shared layer? (Both teams, or a central data platform team.)
- Define a **data contract** for the shared layer:
  - Schema, expected columns, data types.
  - Freshness SLA (e.g., available by 6 AM daily).
  - Change management process: any schema change requires approval from both teams.
- Define change management: any change to the shared layer requires notification and review.

**Step 5: Implement**
- Build the shared silver layer.
- Migrate both teams' pipelines to consume from the silver layer.
- Validate that outputs match the old pipelines.
- Decommission the redundant parts of the old pipelines.

**Step 6: Prevent Future Duplication**
- Maintain a **pipeline registry** where teams register their pipelines and source/target tables.
- Before building a new pipeline, search the registry for existing pipelines on the same source.
- Encourage a culture of code sharing and reuse (shared libraries, common utility functions).
- Regular architecture reviews to identify redundancy.

**Benefits of This Approach:**
- Reduces compute cost (one read of the source instead of two).
- Ensures consistency in shared transformations.
- Each team retains autonomy for their specific logic.
- Easier maintenance — fix a bug in the shared layer once, both teams benefit.
- Clear ownership and accountability.